# Uplift Modeling

This notebook is the central modeling notebook for the CRITEO-UPLIFTv2.1 causal
uplift study. It reads the frozen `f0`-`f11` feature contract, the sealed
70/15/15 train/validation/held-out split, and the frozen no-op preprocessing
transform established by the prior notebook, and evaluates uplift-ranking
methods through the frozen metric contract only.

Sections are added as each method's real, executed results exist -- this is a
running record, not a template with placeholder results. As of this run, T07
(Random reference + Response LightGBM baseline) is in progress: its
correctness/artifact-mechanism SMOKE rehearsal (D30) has executed; its FULL,
authoritative development run has not yet been authorized.


In [1]:
import hashlib
import json
import subprocess
import sys
from pathlib import Path


def _find_repo_root(start):
    for candidate in (start, *start.parents):
        if (candidate / 'src' / 'data.py').is_file():
            return candidate
    raise RuntimeError(f'Could not locate repository root above {start}')


WORKING_DIRECTORY = Path.cwd().resolve()
REPO_ROOT = _find_repo_root(WORKING_DIRECTORY)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

NOTEBOOK_PATH = REPO_ROOT / 'kaggle' / '02_uplift_modeling.ipynb'
T05_CONFIG_PATH = REPO_ROOT / 'configs' / 't05_split.json'
T04_CONFIG_PATH = REPO_ROOT / 'configs' / 't04_preprocessing.json'
T07_CONFIG_PATH = REPO_ROOT / 'configs' / 't07_baselines.json'
SELECTOR_PATH = REPO_ROOT / 'configs' / 'data_manifest.json'


def notebook_source_sha256(path):
    payload = json.loads(path.read_text(encoding='utf-8'))
    source_only = [{'cell_type': c['cell_type'], 'source': ''.join(c.get('source', []))} for c in payload['cells']]
    return hashlib.sha256(json.dumps(source_only, sort_keys=True, separators=(',', ':')).encode()).hexdigest()


## 0. Environment and reproducibility setup

In [2]:
import importlib

import lightgbm as lgb

from src.lightgbm_baseline import FROZEN_LIGHTGBM_VERSION

if lgb.__version__ != FROZEN_LIGHTGBM_VERSION:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', f'lightgbm=={FROZEN_LIGHTGBM_VERSION}'])
    raise RuntimeError(
        f'LightGBM was {lgb.__version__}; installed {FROZEN_LIGHTGBM_VERSION}. '
        'A hot importlib.reload() is not sufficient reproducibility enforcement for an '
        'already-imported compiled extension module -- restart the kernel and re-run this '
        'notebook from the top so the freshly-installed binary is what actually loads.'
    )

assert lgb.__version__ == FROZEN_LIGHTGBM_VERSION, (
    f'LightGBM version mismatch after guard: runtime={lgb.__version__}, frozen={FROZEN_LIGHTGBM_VERSION}. '
    'Refusing to fit on an unverified version.'
)
print(f'LightGBM verified: {lgb.__version__}')


LightGBM verified: 4.7.0


In [3]:
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import psutil
import sklearn
from sklearn.model_selection import train_test_split

from src.data import (
    DataContractError, FEATURE_COLUMNS, PROCESSED_COLUMNS, SOURCE_ROW_ID, TREATMENT_COLUMN, PRIMARY_OUTCOME,
    assert_model_feature_contract, finalize_artifact_manifest, load_selector,
    materialize_pandas, open_processed_dataset, sha256_file, write_bytes_new,
    write_json_new, write_text_new,
)
from src.split import SplitDataset, SplitContractError, membership_hash
from src.preprocessing import IdentityFeatureTransform, preprocessing_contract
import src.metrics as metrics
import src.lightgbm_baseline as lgb_baseline

process = psutil.Process()
notebook_baseline_rss_bytes = process.memory_info().rss
print('Setup complete.')


Setup complete.


## 1. Modeling Objective

Two non-causal reference points are established before any causal estimator is
trained (D11, D12, D27): a **Random reference** (a seeded ranking independent
of `X`, `T`, and `Y`, giving the no-skill floor every method is compared
against), and a **Response LightGBM baseline** (`P(Y=1|X)`, a plain
factual-outcome classifier that never sees `T`).

Response probability and treatment effect are different quantities. A
"sure thing" -- someone who converts whether or not they are treated -- ranks
high on response but contributes zero incremental value; a "persuadable" --
who converts only if treated -- can rank low on response despite being exactly
who targeting should reach. A response model with strong AUC can therefore
still rank poorly on uplift, and that is the expected, reportable finding, not
an error to fix. Response's own diagnostics (ROC-AUC, average precision, log
loss) are evaluated separately from its uplift-ranking performance and never
select a causal winner (D27).

Both methods' scores are routed through the same frozen T06 metric interface
(`src/metrics.py`) used by every later estimator, so comparisons are on equal
footing from the start.


## 2. Data, Split & Preprocessing Contracts

In [4]:
t05_config = json.loads(T05_CONFIG_PATH.read_text(encoding='utf-8'))
t04_config = json.loads(T04_CONFIG_PATH.read_text(encoding='utf-8'))
t07_config = json.loads(T07_CONFIG_PATH.read_text(encoding='utf-8'))

assert t05_config['lifecycle_state'] == 'T05_SPLIT_ACCEPTED', t05_config['lifecycle_state']
assert t04_config['lifecycle_state'] == 'T04_ACCEPTED', t04_config['lifecycle_state']

t05_run_manifest_path = REPO_ROOT / t05_config['lifecycle_state_evidence']['authorizing_run_manifest']
t05_run_root = t05_run_manifest_path.parent.parent
membership = pd.read_csv(t05_run_root / 'audit' / 'split_membership.csv')
observed_membership_hash = membership_hash(membership)
expected_membership_hash = t05_config['lifecycle_state_evidence']['membership_sha256']
if observed_membership_hash != expected_membership_hash:
    raise SplitContractError(
        f'Split membership hash mismatch: expected {expected_membership_hash}, observed {observed_membership_hash}'
    )

split_dataset = SplitDataset(membership=membership)
train_ids_full = split_dataset.train_ids()
validation_ids_full = split_dataset.validation_ids()
print(f'Split membership verified. train={len(train_ids_full):,} validation={len(validation_ids_full):,}')


Split membership verified. train=9,785,714 validation=2,096,938


In [5]:
selector = load_selector(SELECTOR_PATH, REPO_ROOT)
dataset = open_processed_dataset(selector)
processed_sha256 = selector.payload['processed_sha256']

full_frame = materialize_pandas(dataset, columns=PROCESSED_COLUMNS, row_limit=None)
if tuple(full_frame.columns) != PROCESSED_COLUMNS:
    raise RuntimeError('Processed frame column order drifted from the frozen contract')

print(f'Processed dataset loaded: {len(full_frame):,} rows, sha256={processed_sha256[:16]}...')


Processed dataset loaded: 13,979,592 rows, sha256=fd2739bb074a50fa...


In [6]:
RUN_T07_STAGE = False  # T07 (Random reference + Response LightGBM baseline) is already accepted.
T07_FULL_RUN_ID_ACCEPTED = 't07_full_20260818T111446Z_593205'
T07_ERRATUM_RUN_ID_ACCEPTED = 't07_audit_erratum_20260818T150917Z_995999'
T07_T08_RANDOM_LABEL_ERRATUM_RUN_ID_ACCEPTED = 't07_t08_random_label_erratum_20260818T162445Z_587615'

if not RUN_T07_STAGE:
    t07_full_root = REPO_ROOT / 'outputs' / 'runs' / T07_FULL_RUN_ID_ACCEPTED
    t07_manifest_path = t07_full_root / 'audit' / 'artifact_manifest.json'
    if not t07_manifest_path.is_file():
        raise RuntimeError(
            f'RUN_T07_STAGE is False, but the accepted T07 FULL run evidence was not found at '
            f'{t07_full_root}. This environment does not have the governed T07 evidence mounted. '
            f'Refusing to silently set RUN_T07_STAGE = True and recompute T07 -- either provide/mount '
            f'the accepted governed run evidence (outputs/runs/{T07_FULL_RUN_ID_ACCEPTED}/), or '
            f'explicitly set RUN_T07_STAGE = True in this cell to opt into reproducing T07 from scratch.'
        )
    t07_manifest = json.loads(t07_manifest_path.read_text(encoding='utf-8'))
    t07_artifact_hashes = {a['path']: a['sha256'] for a in t07_manifest['artifacts']}
    t07_model_summary_path = t07_full_root / 'tables' / 'model_summary.csv'
    t07_model_summary_actual_sha256 = hashlib.sha256(t07_model_summary_path.read_bytes()).hexdigest()
    t07_model_summary_expected_sha256 = t07_artifact_hashes.get('tables/model_summary.csv')
    if t07_model_summary_expected_sha256 is None or t07_model_summary_actual_sha256 != t07_model_summary_expected_sha256:
        raise RuntimeError(
            f'T07 FULL tables/model_summary.csv hash does not match its own artifact manifest '
            f'(expected {t07_model_summary_expected_sha256}, actual {t07_model_summary_actual_sha256}). '
            f'Refusing to reuse unverified evidence.'
        )
    t07_reference_summary = pd.read_csv(t07_model_summary_path)
    # Label erratum (audit-only; numeric values unchanged, see
    # T07_T08_RANDOM_LABEL_ERRATUM_RUN_ID_ACCEPTED): the 'random' row reports the seed-42
    # illustrative draw, not the theoretical no-skill reference -- relabel in memory only,
    # the source file on disk is never touched.
    t07_reference_summary = t07_reference_summary.copy()
    t07_reference_summary.loc[t07_reference_summary['ranking_method'] == 'random', 'ranking_method'] = 'random_seed_42'
    t07_theoretical_random_qini_area = float(t07_reference_summary['theoretical_random_qini_area'].iloc[0])
    print('Using hash-verified frozen development results (T07).')
    print(f'  Reproducibility: T07 FULL run_id={T07_FULL_RUN_ID_ACCEPTED}')
else:
    print('RUN_T07_STAGE = True: T07 SMOKE and FULL will be recomputed from scratch below.')


Using hash-verified frozen development results (T07).
  Reproducibility: T07 FULL run_id=t07_full_20260818T111446Z_593205


## 3. Scale-Gating (D30): SMOKE Verification

Under D30, T07 uses `SMOKE -> FULL` with `resource_gates = 0` (recorded and
justified in `configs/t07_baselines.json`: the full-scale data path is already
proven by T01, and a single Response LightGBM binary classifier over 12
numeric features does not meet D30's resource-risk trigger). SMOKE is a
bounded, development-only rehearsal of correctness, the feature/leakage
contract, row alignment, serialization/reload, and artifact mechanics -- it
never supports a performance claim or selects a model/config/seed. This
section executes SMOKE only; FULL is a separate, later-authorized run.


*(This section -- T07 Random reference and Response LightGBM baseline -- is already accepted. It is skipped by default on Run All; see `RUN_T07_STAGE` immediately above. Set it to `True` only to deliberately reproduce T07 from scratch.)*

In [7]:
if RUN_T07_STAGE:
    SMOKE_SIZE = t07_config['scale_gating']['smoke_size']
    SMOKE_SEED = t07_config['scale_gating']['smoke_seed']
    RESOURCE_GATES = t07_config['scale_gating']['resource_gates']
    RUN_FULL_STAGE = True  # FULL authorized this run under D30 (resource_gates=0), after the SMOKE PASS above.

    smoke_started = datetime.now(timezone.utc)
    smoke_wall_start = __import__('time').perf_counter()

    RUN_ID = smoke_started.strftime('t07_smoke_%Y%m%dT%H%M%SZ_%f')
    RUN_ROOT = REPO_ROOT / 'outputs' / 'runs' / RUN_ID
    RUN_ROOT.mkdir(parents=True, exist_ok=False)

    try:
        git_head = subprocess.run(['git', 'rev-parse', 'HEAD'], cwd=REPO_ROOT, check=True, capture_output=True, text=True).stdout.strip()
        git_dirty = bool(subprocess.run(['git', 'status', '--porcelain'], cwd=REPO_ROOT, check=True, capture_output=True, text=True).stdout.strip())
    except (OSError, subprocess.CalledProcessError):
        git_head, git_dirty = None, None

    print(f'RUN_ID = {RUN_ID}')
    print(f'resource_gates = {RESOURCE_GATES}, smoke_size = {SMOKE_SIZE}, seed = {SMOKE_SEED}')


In [8]:
if RUN_T07_STAGE:
    def _joint_strata(frame, id_column, treatment_column, outcome_column):
        return frame[treatment_column].astype(str) + '_' + frame[outcome_column].astype(str)


    def smoke_sample_partition(partition_ids, quota, full_frame, seed):
        # One deterministic joint-(T,Y)-stratified draw of `quota` rows from
        # `partition_ids`, reusing the same train_test_split stratification
        # mechanism already used by src/split.py's assign_split() -- no new
        # sampling algorithm, no reusable scale-rung module.
        subset = full_frame.loc[full_frame[SOURCE_ROW_ID].isin(partition_ids)]
        strata = _joint_strata(subset, SOURCE_ROW_ID, TREATMENT_COLUMN, PRIMARY_OUTCOME)
        selected_ids, _ = train_test_split(
            subset[SOURCE_ROW_ID].to_numpy(), train_size=quota, random_state=seed, stratify=strata,
        )
        return np.sort(selected_ids)


    p_train = len(train_ids_full) / (len(train_ids_full) + len(validation_ids_full))
    train_quota = round(SMOKE_SIZE * p_train)
    validation_quota = SMOKE_SIZE - train_quota

    smoke_train_ids = smoke_sample_partition(train_ids_full, train_quota, full_frame, SMOKE_SEED)
    smoke_validation_ids = smoke_sample_partition(validation_ids_full, validation_quota, full_frame, SMOKE_SEED)

    # Held-out isolation is proved positively, by construction, from the two
    # sanctioned development partitions only (`train_ids_full`/`validation_ids_full`,
    # obtained solely via SplitDataset.train_ids()/.validation_ids()). This never
    # reads split_membership.csv's held_out label, never calls
    # SplitDataset.held_out_ids(), and never inspects any held-out row, ID,
    # feature, label, or summary through any path -- the held-out partition is
    # simply absent from every set these assertions reference.
    smoke_total = len(smoke_train_ids) + len(smoke_validation_ids)
    development_ids_full = set(train_ids_full) | set(validation_ids_full)
    assert smoke_total == SMOKE_SIZE, f'SMOKE total {smoke_total} != {SMOKE_SIZE}'
    assert set(smoke_train_ids).issubset(set(train_ids_full)), 'smoke_train_ids must be a subset of the frozen train partition'
    assert set(smoke_validation_ids).issubset(set(validation_ids_full)), 'smoke_validation_ids must be a subset of the frozen validation partition'
    assert set(smoke_train_ids).isdisjoint(set(smoke_validation_ids)), 'smoke_train_ids and smoke_validation_ids must be disjoint'
    assert (set(smoke_train_ids) | set(smoke_validation_ids)).issubset(development_ids_full), (
        'every SMOKE-selected ID must belong to train_ids_full union validation_ids_full'
    )

    print(f'SMOKE: train={len(smoke_train_ids):,} validation={len(smoke_validation_ids):,} total={smoke_total:,}')
    print('Held-out isolation: proved by construction from train_ids_full/validation_ids_full only; held-out never read.')


In [9]:
if RUN_T07_STAGE:
    def joint_ty_support(ids, full_frame):
        subset = full_frame.loc[full_frame[SOURCE_ROW_ID].isin(ids)]
        counts = subset.groupby([TREATMENT_COLUMN, PRIMARY_OUTCOME], observed=True).size()
        counts = counts.reindex(pd.MultiIndex.from_product([[0, 1], [0, 1]], names=[TREATMENT_COLUMN, PRIMARY_OUTCOME]), fill_value=0)
        return {f'T={t},Y={y}': int(n) for (t, y), n in counts.items()}


    smoke_train_support = joint_ty_support(smoke_train_ids, full_frame)
    smoke_validation_support = joint_ty_support(smoke_validation_ids, full_frame)
    smoke_all_cells_nonempty = all(n > 0 for n in {**smoke_train_support, **smoke_validation_support}.values())
    print('SMOKE train (T,Y) support:', smoke_train_support)
    print('SMOKE validation (T,Y) support:', smoke_validation_support)
    print('All joint-(T,Y) cells non-empty:', smoke_all_cells_nonempty)

    smoke_sample_ids_frame = pd.concat([
        pd.DataFrame({SOURCE_ROW_ID: smoke_train_ids, 'partition': 'train'}),
        pd.DataFrame({SOURCE_ROW_ID: smoke_validation_ids, 'partition': 'validation'}),
    ], ignore_index=True)
    smoke_sample_ids_sha256 = hashlib.sha256(
        smoke_sample_ids_frame.sort_values(SOURCE_ROW_ID)[SOURCE_ROW_ID].to_numpy(dtype='<i8').tobytes()
    ).hexdigest()

    write_bytes_new(RUN_ROOT, 'audit/smoke_sample_row_ids.parquet', smoke_sample_ids_frame.to_parquet(index=False))
    write_json_new(RUN_ROOT, 'audit/smoke_sample_manifest.json', {
        'run_id': RUN_ID,
        'stage': 't07_smoke',
        'population': 'smoke_50000_rows',
        'smoke_size': SMOKE_SIZE,
        'smoke_seed': SMOKE_SEED,
        'train_quota': int(train_quota),
        'validation_quota': int(validation_quota),
        'train_count': int(len(smoke_train_ids)),
        'validation_count': int(len(smoke_validation_ids)),
        'total_count': int(smoke_total),
        'train_support': smoke_train_support,
        'validation_support': smoke_validation_support,
        'all_joint_ty_cells_nonempty': bool(smoke_all_cells_nonempty),
        'smoke_sample_row_ids_sha256': smoke_sample_ids_sha256,
        'held_out_isolation_method': (
            'guaranteed_by_construction_from_sanctioned_development_partitions: smoke_train_ids and '
            'smoke_validation_ids are proved subsets of train_ids_full/validation_ids_full (each obtained '
            'solely via SplitDataset.train_ids()/.validation_ids()), pairwise disjoint, and their union is a '
            'subset of train_ids_full union validation_ids_full -- held-out is never read via '
            'split_membership.csv\'s held_out label, SplitDataset.held_out_ids(), or any other path'
        ),
    })
    print('SMOKE sample identity persisted.')


In [10]:
if RUN_T07_STAGE:
    smoke_train_frame = full_frame.loc[full_frame[SOURCE_ROW_ID].isin(smoke_train_ids)].sort_values(SOURCE_ROW_ID).reset_index(drop=True)
    smoke_validation_frame = full_frame.loc[full_frame[SOURCE_ROW_ID].isin(smoke_validation_ids)].sort_values(SOURCE_ROW_ID).reset_index(drop=True)

    transform = IdentityFeatureTransform()
    X_smoke_train = transform.fit_transform(smoke_train_frame)
    X_smoke_validation = transform.transform(smoke_validation_frame)
    assert_model_feature_contract(X_smoke_train.columns)
    assert_model_feature_contract(X_smoke_validation.columns)

    y_smoke_train = smoke_train_frame[PRIMARY_OUTCOME].astype('float64')
    y_smoke_validation = smoke_validation_frame[PRIMARY_OUTCOME].astype('float64')
    t_smoke_validation = smoke_validation_frame[TREATMENT_COLUMN].astype('float64').to_numpy()
    y_smoke_validation_arr = y_smoke_validation.to_numpy()
    source_row_id_smoke_validation = smoke_validation_frame[SOURCE_ROW_ID].to_numpy()

    print(f'X_smoke_train: {X_smoke_train.shape}, X_smoke_validation: {X_smoke_validation.shape}')
    print('Feature contract verified: X is exactly', tuple(X_smoke_train.columns))


### 3.1 Response LightGBM baseline (SMOKE)

In [11]:
if RUN_T07_STAGE:
    response_model = lgb_baseline.fit_binary_classifier(
        X_smoke_train, y_smoke_train, X_smoke_validation, y_smoke_validation,
    )
    response_probabilities_smoke = lgb_baseline.predict_probabilities(response_model, X_smoke_validation)

    assert np.isfinite(response_probabilities_smoke).all()
    assert (response_probabilities_smoke >= 0).all() and (response_probabilities_smoke <= 1).all()
    assert len(response_probabilities_smoke) == len(smoke_validation_ids)
    assert set(source_row_id_smoke_validation) == set(smoke_validation_ids)

    print(f'Response fit complete. best_iteration={response_model.best_iteration}, config_hash={response_model.config_hash[:16]}...')
    print('Predictions bounded/finite/aligned: OK')


### 3.2 Random reference (SMOKE) -- via the public T06 interface only

In [12]:
if RUN_T07_STAGE:
    random_scores_smoke = metrics.seeded_random_scores(len(smoke_validation_ids), seed=metrics.RANDOM_RANKING_SEED)
    random_ranking_smoke = metrics.evaluate_ranking(
        random_scores_smoke, t_smoke_validation, y_smoke_validation_arr, source_row_id_smoke_validation,
    )
    random_reference_distribution_smoke = metrics.random_ranking_reference_distribution(
        t_smoke_validation, y_smoke_validation_arr, source_row_id_smoke_validation,
    )
    assert len(random_reference_distribution_smoke) == metrics.RANDOM_RANKING_REFERENCE_DRAWS == 200

    response_ranking_smoke = metrics.evaluate_ranking(
        response_probabilities_smoke, t_smoke_validation, y_smoke_validation_arr, source_row_id_smoke_validation,
    )
    response_diag_smoke = metrics.response_diagnostics(response_probabilities_smoke, y_smoke_validation_arr)
    ate_smoke = metrics.compute_ate(t_smoke_validation, y_smoke_validation_arr)

    print(f'Random: 1 illustrative draw + {len(random_reference_distribution_smoke)} reference draws computed via the public T06 interface.')
    print(f'Response ranking qini_above_random (SMOKE, non-substantive): {response_ranking_smoke.qini_above_random:.6f}')


### 3.3 SMOKE artifacts

In [13]:
if RUN_T07_STAGE:
    def _rows_with_run_context(rows, population='smoke_50000_rows'):
        for row in rows:
            row = dict(row)
            row.setdefault('run_id', RUN_ID)
            row.setdefault('stage', 't07_smoke')
            row.setdefault('population', population)
            yield row


    ate_summary_rows = list(_rows_with_run_context([{'method': 'assigned_arm', **ate_smoke.__dict__}]))
    response_diagnostics_rows = list(_rows_with_run_context([{'method': 'response', **response_diag_smoke.__dict__}]))

    uplift_at_k_rows = []
    for label in metrics.RANKING_K_LABELS:
        uplift_at_k_rows.append({'method': 'random', 'k': label, 'uplift': random_ranking_smoke.uplift_at_k[label],
                                  'incremental_conversions': random_ranking_smoke.incremental_conversions_at_k[label],
                                  'status': random_ranking_smoke.top_k_status[label]})
        uplift_at_k_rows.append({'method': 'response', 'k': label, 'uplift': response_ranking_smoke.uplift_at_k[label],
                                  'incremental_conversions': response_ranking_smoke.incremental_conversions_at_k[label],
                                  'status': response_ranking_smoke.top_k_status[label]})
    uplift_at_k_rows = list(_rows_with_run_context(uplift_at_k_rows))

    model_summary_rows = []
    for name, result in (('random', random_ranking_smoke), ('response', response_ranking_smoke)):
        model_summary_rows.append({
            'ranking_method': name,
            'qini_area': result.qini_area,
            'theoretical_random_qini_area': result.theoretical_random_qini_area,
            'qini_above_random': result.qini_above_random,
            'qini_above_random_permutation': random_ranking_smoke.qini_above_random,
            'uplift_at_10pct': result.uplift_at_k['10pct'],
            'uplift_at_20pct': result.uplift_at_k['20pct'],
            'uplift_at_30pct': result.uplift_at_k['30pct'],
            'incremental_conversions_at_10pct': result.incremental_conversions_at_k['10pct'],
            'incremental_conversions_at_20pct': result.incremental_conversions_at_k['20pct'],
            'incremental_conversions_at_30pct': result.incremental_conversions_at_k['30pct'],
        })
    model_summary_rows = list(_rows_with_run_context(model_summary_rows))

    random_deciles_rows = list(_rows_with_run_context(random_ranking_smoke.decile_table.to_dict('records')))
    response_deciles_rows = list(_rows_with_run_context(response_ranking_smoke.decile_table.to_dict('records')))

    for name, rows in (
        ('tables/ate_summary.csv', ate_summary_rows),
        ('tables/response_diagnostics.csv', response_diagnostics_rows),
        ('tables/random_deciles.csv', random_deciles_rows),
        ('tables/response_deciles.csv', response_deciles_rows),
        ('tables/uplift_at_k.csv', uplift_at_k_rows),
        ('tables/model_summary.csv', model_summary_rows),
    ):
        write_text_new(RUN_ROOT, name, pd.DataFrame(rows).to_csv(index=False, lineterminator='\n'))

    print('SMOKE tables written.')


In [14]:
if RUN_T07_STAGE:
    response_predictions_frame = pd.DataFrame({
        SOURCE_ROW_ID: source_row_id_smoke_validation,
        'response_probability': response_probabilities_smoke,
    })
    random_scores_frame = pd.DataFrame({
        SOURCE_ROW_ID: source_row_id_smoke_validation,
        'random_score': random_scores_smoke,
    })
    response_predictions_bytes = response_predictions_frame.to_parquet(index=False)
    random_scores_bytes = random_scores_frame.to_parquet(index=False)

    write_bytes_new(RUN_ROOT, 'predictions/development/response/seed_42/validation_predictions.parquet', response_predictions_bytes)
    write_bytes_new(RUN_ROOT, 'predictions/development/random/seed_42/validation_scores.parquet', random_scores_bytes)

    response_model_text = response_model.booster.model_to_string()
    write_text_new(RUN_ROOT, 'models/response_model.txt', response_model_text)

    random_baseline_summary_rows = list(_rows_with_run_context([{
        'theoretical_random_qini_area': random_ranking_smoke.theoretical_random_qini_area,
        'illustrative_draw_seed': metrics.RANDOM_RANKING_SEED,
        'illustrative_draw_qini_area': random_ranking_smoke.qini_area,
        'illustrative_draw_qini_above_random': random_ranking_smoke.qini_above_random,
    }]))
    write_text_new(RUN_ROOT, 'audit/random_baseline_summary.csv', pd.DataFrame(random_baseline_summary_rows).to_csv(index=False, lineterminator='\n'))

    random_baseline_draws_rows = list(_rows_with_run_context(random_reference_distribution_smoke.to_dict('records')))
    write_text_new(RUN_ROOT, 'audit/random_baseline_draws.csv', pd.DataFrame(random_baseline_draws_rows).to_csv(index=False, lineterminator='\n'))

    model_probability_diagnostics_rows = list(_rows_with_run_context([{'method': 'response', **response_diag_smoke.__dict__}]))
    write_text_new(RUN_ROOT, 'audit/model_probability_diagnostics.csv', pd.DataFrame(model_probability_diagnostics_rows).to_csv(index=False, lineterminator='\n'))

    definitions = metrics.metric_definitions()
    definitions_sha256 = hashlib.sha256(json.dumps(definitions, sort_keys=True).encode()).hexdigest()

    def _flatten(value):
        return value if isinstance(value, (str, int, float, bool)) or value is None else json.dumps(value)

    definitions_rows = list(_rows_with_run_context(
        [{'field': k, 'value': _flatten(v)} for k, v in definitions.items()]
        + [{'field': 'definitions_sha256', 'value': definitions_sha256}]
    ))
    write_text_new(RUN_ROOT, 'audit/metric_definitions.csv', pd.DataFrame(definitions_rows).to_csv(index=False, lineterminator='\n'))

    response_predictions_sha256 = hashlib.sha256(response_predictions_bytes).hexdigest()
    random_scores_sha256 = hashlib.sha256(random_scores_bytes).hexdigest()
    print('SMOKE audit/model/prediction artifacts written.')


### 3.4 SMOKE reproducibility check (T07.7, SMOKE scope)

In [15]:
if RUN_T07_STAGE:
    reloaded_booster = lgb.Booster(model_str=response_model_text)
    reloaded_config_hash = lgb_baseline.config_hash()
    config_hash_matches = reloaded_config_hash == response_model.config_hash

    X_smoke_validation_rebuilt = transform.transform(smoke_validation_frame)
    reloaded_probabilities = np.asarray(reloaded_booster.predict(X_smoke_validation_rebuilt, num_iteration=response_model.best_iteration), dtype=np.float64)
    prediction_reload_matches = np.allclose(reloaded_probabilities, response_probabilities_smoke, rtol=1e-6, atol=1e-8)

    row_identity_matches = set(source_row_id_smoke_validation) == set(smoke_validation_ids)

    random_scores_regenerated = metrics.seeded_random_scores(len(smoke_validation_ids), seed=metrics.RANDOM_RANKING_SEED)
    random_scores_exact_match = np.array_equal(random_scores_regenerated, random_scores_smoke)

    reload_verification = {
        'config_hash_matches': bool(config_hash_matches),
        'prediction_reload_matches_within_tolerance': bool(prediction_reload_matches),
        'row_identity_matches': bool(row_identity_matches),
        'random_scores_exact_match': bool(random_scores_exact_match),
        'tolerance': {'rtol': 1e-6, 'atol': 1e-8},
    }
    print(reload_verification)
    assert all(reload_verification[k] for k in ('config_hash_matches', 'prediction_reload_matches_within_tolerance', 'row_identity_matches', 'random_scores_exact_match'))


In [16]:
if RUN_T07_STAGE:
    smoke_wall_seconds = __import__('time').perf_counter() - smoke_wall_start
    smoke_peak_rss_bytes = process.memory_info().rss

    write_json_new(RUN_ROOT, 'audit/environment.json', {
        'run_id': RUN_ID,
        'created_at_utc': smoke_started.isoformat(),
        'git_head': git_head,
        'git_dirty': git_dirty,
        'python': sys.version,
        'numpy': np.__version__,
        'pandas': pd.__version__,
        'scikit_learn': sklearn.__version__,
        'lightgbm': lgb.__version__,
        'psutil': psutil.__version__,
        'total_ram_bytes': psutil.virtual_memory().total,
    })

    write_json_new(RUN_ROOT, 'audit/run_config.json', {
        'run_id': RUN_ID,
        'created_at_utc': smoke_started.isoformat(),
        'stage': 't07_smoke',
        'population': 'smoke_50000_rows',
        'git_head': git_head,
        'git_dirty': git_dirty,
        'real_or_held_out_data_accessed': False,
        'notebook_source_sha256': notebook_source_sha256(NOTEBOOK_PATH) if NOTEBOOK_PATH.is_file() else None,
        'src_lightgbm_baseline_sha256': sha256_file(REPO_ROOT / 'src' / 'lightgbm_baseline.py'),
        'src_metrics_sha256': sha256_file(REPO_ROOT / 'src' / 'metrics.py'),
        'processed_sha256': processed_sha256,
        'split_membership_sha256': observed_membership_hash,
        't05_run_id': t05_config['lifecycle_state_evidence']['authorizing_run_id'],
        't04_lifecycle_state': t04_config['lifecycle_state'],
        't06_authoritative_run_id': t07_config['input']['metrics_interface']['authoritative_t06_run_id'],
        't06_authoritative_definitions_sha256': t07_config['input']['metrics_interface']['authoritative_t06_definitions_sha256'],
        'scale_gating': {'policy': 'D30', 'stage': 'SMOKE', 'resource_gates': RESOURCE_GATES, 'smoke_size': SMOKE_SIZE, 'smoke_seed': SMOKE_SEED},
        'lightgbm_config': lgb_baseline.FROZEN_BINARY_CONFIG,
        'lightgbm_config_hash': response_model.config_hash,
        'lightgbm_best_iteration': response_model.best_iteration,
        'lightgbm_num_boost_round_cap': lgb_baseline.NUM_BOOST_ROUND_CAP,
        'lightgbm_early_stopping_rounds': lgb_baseline.EARLY_STOPPING_ROUNDS,
        'response_predictions_sha256': response_predictions_sha256,
        'random_scores_sha256': random_scores_sha256,
        'definitions_sha256': definitions_sha256,
        'resource_evidence': {
            'wall_seconds': smoke_wall_seconds,
            'baseline_rss_bytes': notebook_baseline_rss_bytes,
            'peak_rss_bytes': smoke_peak_rss_bytes,
            'peak_rss_delta_bytes': smoke_peak_rss_bytes - notebook_baseline_rss_bytes,
        },
        'reload_verification': reload_verification,
    })

    print(f'SMOKE resource evidence: wall_seconds={smoke_wall_seconds:.2f}, peak_rss_delta_bytes={smoke_peak_rss_bytes - notebook_baseline_rss_bytes:,}')


In [17]:
if RUN_T07_STAGE:
    finalize_artifact_manifest(
        RUN_ROOT,
        run_id=RUN_ID,
        final_status='COMPLETED_T07_SMOKE_VERIFIED',
        created_at_utc=datetime.now(timezone.utc).isoformat(),
        stage='t07_smoke',
        population='smoke_50000_rows',
        external_artifacts=[
            {'path': 'kaggle/02_uplift_modeling.ipynb#sources', 'role': 'human_readable_protocol_source',
             'sha256': notebook_source_sha256(NOTEBOOK_PATH) if NOTEBOOK_PATH.is_file() else None, 'status': 'PASS'},
            {'path': 'src/lightgbm_baseline.py', 'role': 'reusable_t07_contract', 'sha256': sha256_file(REPO_ROOT / 'src' / 'lightgbm_baseline.py'), 'status': 'PASS'},
            {'path': 'src/metrics.py', 'role': 'reusable_t06_contract', 'sha256': sha256_file(REPO_ROOT / 'src' / 'metrics.py'), 'status': 'PASS'},
        ],
    )
    print(f'SMOKE run finalized: {RUN_ID}')

    immutable_write_refused = False
    try:
        write_json_new(RUN_ROOT, 'audit/should_be_refused.json', {'x': 1})
    except (FileExistsError, DataContractError):
        immutable_write_refused = True
    except Exception:
        immutable_write_refused = False
    print(f'Immutable-run write refusal verified: {immutable_write_refused}')
    assert immutable_write_refused


### 3.5 SMOKE summary

In [18]:
if RUN_T07_STAGE:
    smoke_summary = {
        'run_id': RUN_ID,
        'smoke_total': int(smoke_total),
        'smoke_train_count': int(len(smoke_train_ids)),
        'smoke_validation_count': int(len(smoke_validation_ids)),
        'all_joint_ty_cells_nonempty': bool(smoke_all_cells_nonempty),
        'lightgbm_version': lgb.__version__,
        'lightgbm_config_hash': response_model.config_hash,
        'best_iteration': response_model.best_iteration,
        'random_reference_draws': len(random_reference_distribution_smoke),
        'reload_verification': reload_verification,
        'resource_wall_seconds': smoke_wall_seconds,
        'immutable_write_refused': immutable_write_refused,
    }
    print(json.dumps(smoke_summary, indent=2))


#### Reproducibility note (technical): T07 FULL execution

SMOKE passed every correctness/artifact-mechanism check in the section above.
Under D30, T07's approved path is `SMOKE -> FULL` with `resource_gates = 0`
(recorded and justified in `configs/t07_baselines.json`), so FULL is the next
and only remaining stage. FULL fits on the **complete** frozen train partition,
early-stops on the **complete** frozen validation partition, and is the
authoritative development run whose results the sections below report.
Held-out remains completely sealed throughout -- this section never reads
`SplitDataset.held_out_ids()` or any held-out row.

In [19]:
if RUN_T07_STAGE:
    from IPython.display import Markdown, display
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt

    full_started = datetime.now(timezone.utc)
    full_wall_start = __import__('time').perf_counter()
    full_baseline_rss_bytes = process.memory_info().rss

    FULL_RUN_ID = full_started.strftime('t07_full_%Y%m%dT%H%M%SZ_%f')
    FULL_RUN_ROOT = REPO_ROOT / 'outputs' / 'runs' / FULL_RUN_ID
    FULL_RUN_ROOT.mkdir(parents=True, exist_ok=False)

    try:
        full_git_head = subprocess.run(['git', 'rev-parse', 'HEAD'], cwd=REPO_ROOT, check=True, capture_output=True, text=True).stdout.strip()
        full_git_dirty = bool(subprocess.run(['git', 'status', '--porcelain'], cwd=REPO_ROOT, check=True, capture_output=True, text=True).stdout.strip())
    except (OSError, subprocess.CalledProcessError):
        full_git_head, full_git_dirty = None, None

    print(f'FULL_RUN_ID = {FULL_RUN_ID}')


In [20]:
if RUN_T07_STAGE:
    full_train_frame = full_frame.loc[full_frame[SOURCE_ROW_ID].isin(train_ids_full)].sort_values(SOURCE_ROW_ID).reset_index(drop=True)
    full_validation_frame = full_frame.loc[full_frame[SOURCE_ROW_ID].isin(validation_ids_full)].sort_values(SOURCE_ROW_ID).reset_index(drop=True)
    assert len(full_train_frame) == len(train_ids_full)
    assert len(full_validation_frame) == len(validation_ids_full)

    transform_full = IdentityFeatureTransform()
    X_full_train = transform_full.fit_transform(full_train_frame)
    X_full_validation = transform_full.transform(full_validation_frame)
    assert_model_feature_contract(X_full_train.columns)
    assert_model_feature_contract(X_full_validation.columns)

    y_full_train = full_train_frame[PRIMARY_OUTCOME].astype('float64')
    y_full_validation = full_validation_frame[PRIMARY_OUTCOME].astype('float64')
    t_full_validation = full_validation_frame[TREATMENT_COLUMN].astype('float64').to_numpy()
    y_full_validation_arr = y_full_validation.to_numpy()
    source_row_id_full_validation = full_validation_frame[SOURCE_ROW_ID].to_numpy()

    print(f'FULL: X_train={X_full_train.shape}, X_validation={X_full_validation.shape}')


## 4. Random Reference

In [21]:
if RUN_T07_STAGE:
    random_scores_full = metrics.seeded_random_scores(len(validation_ids_full), seed=metrics.RANDOM_RANKING_SEED)
    random_ranking_full = metrics.evaluate_ranking(
        random_scores_full, t_full_validation, y_full_validation_arr, source_row_id_full_validation,
    )
    random_reference_distribution_full = metrics.random_ranking_reference_distribution(
        t_full_validation, y_full_validation_arr, source_row_id_full_validation,
    )
    assert len(random_reference_distribution_full) == metrics.RANDOM_RANKING_REFERENCE_DRAWS == 200

    display(Markdown(
        f"The theoretical expected-random Qini area on the frozen validation population is "
        f"**{random_ranking_full.theoretical_random_qini_area:.4f}**. One deterministic seed-42 illustrative "
        f"random ranking realizes `qini_area = {random_ranking_full.qini_area:.4f}` "
        f"(`qini_above_random = {random_ranking_full.qini_above_random:.4f}`), and the frozen "
        f"{len(random_reference_distribution_full)}-draw random-ranking reference distribution has "
        f"`qini_above_random` mean **{random_reference_distribution_full['qini_above_random'].mean():.4f}** "
        f"(std {random_reference_distribution_full['qini_above_random'].std():.4f}), consistent with a "
        f"no-skill ranking centered near zero. This distribution is secondary empirical context; the "
        f"theoretical line remains the primary random reference (D11)."
    ))


## 5. Response LightGBM Baseline

In [22]:
if RUN_T07_STAGE:
    response_model_full = lgb_baseline.fit_binary_classifier(
        X_full_train, y_full_train, X_full_validation, y_full_validation,
    )
    response_probabilities_full = lgb_baseline.predict_probabilities(response_model_full, X_full_validation)

    assert np.isfinite(response_probabilities_full).all()
    assert (response_probabilities_full >= 0).all() and (response_probabilities_full <= 1).all()
    assert set(source_row_id_full_validation) == set(validation_ids_full)

    response_diag_full = metrics.response_diagnostics(response_probabilities_full, y_full_validation_arr)
    response_ranking_full = metrics.evaluate_ranking(
        response_probabilities_full, t_full_validation, y_full_validation_arr, source_row_id_full_validation,
    )
    ate_full = metrics.compute_ate(t_full_validation, y_full_validation_arr)

    print(f'Response fit complete. best_iteration={response_model_full.best_iteration}, config_hash={response_model_full.config_hash[:16]}...')


In [23]:
if RUN_T07_STAGE:
    display(Markdown(
        f"**Response diagnostics (factual-outcome prediction quality -- diagnostic only, D27; never a causal "
        f"ranking claim):** ROC-AUC = {response_diag_full.roc_auc:.4f}, average precision = "
        f"{response_diag_full.average_precision:.4f}, log loss = {response_diag_full.log_loss:.4f}.\n\n"
        f"**Response as an uplift ranking** (the same probability scores, evaluated as a policy via the "
        f"identical T06 interface used for every method): `qini_area = {response_ranking_full.qini_area:.4f}`, "
        f"`qini_above_random = {response_ranking_full.qini_above_random:.4f}` "
        f"(theoretical random = {response_ranking_full.theoretical_random_qini_area:.4f}). "
        f"`uplift@10% = {response_ranking_full.uplift_at_k['10pct']}`, "
        f"`uplift@20% = {response_ranking_full.uplift_at_k['20pct']}`, "
        f"`uplift@30% = {response_ranking_full.uplift_at_k['30pct']}` "
        f"(status: {response_ranking_full.top_k_status['10pct']}/{response_ranking_full.top_k_status['20pct']}/{response_ranking_full.top_k_status['30pct']}).\n\n"
        f"**Assigned-arm ATE** (population aggregate, not a ranking estimator; D24 methodology note): "
        f"{ate_full.ate:.4f} ({ate_full.ate_percentage_points:.2f} pp), 95% CI "
        f"[{ate_full.ci_95_low:.4f}, {ate_full.ci_95_high:.4f}]."
    ))


### FULL artifacts

In [24]:
if RUN_T07_STAGE:
    def _full_rows_with_run_context(rows, population='full_development_population'):
        for row in rows:
            row = dict(row)
            row.setdefault('run_id', FULL_RUN_ID)
            row.setdefault('stage', 't07_full')
            row.setdefault('population', population)
            yield row


    ate_summary_rows_full = list(_full_rows_with_run_context([{'method': 'assigned_arm', **ate_full.__dict__}]))
    response_diagnostics_rows_full = list(_full_rows_with_run_context([{'method': 'response', **response_diag_full.__dict__}]))

    uplift_at_k_rows_full = []
    for label in metrics.RANKING_K_LABELS:
        uplift_at_k_rows_full.append({'method': 'random', 'k': label, 'uplift': random_ranking_full.uplift_at_k[label],
                                       'incremental_conversions': random_ranking_full.incremental_conversions_at_k[label],
                                       'status': random_ranking_full.top_k_status[label]})
        uplift_at_k_rows_full.append({'method': 'response', 'k': label, 'uplift': response_ranking_full.uplift_at_k[label],
                                       'incremental_conversions': response_ranking_full.incremental_conversions_at_k[label],
                                       'status': response_ranking_full.top_k_status[label]})
    uplift_at_k_rows_full = list(_full_rows_with_run_context(uplift_at_k_rows_full))

    model_summary_rows_full = []
    for name, result in (('random', random_ranking_full), ('response', response_ranking_full)):
        model_summary_rows_full.append({
            'ranking_method': name,
            'qini_area': result.qini_area,
            'theoretical_random_qini_area': result.theoretical_random_qini_area,
            'qini_above_random': result.qini_above_random,
            'qini_above_random_permutation': random_ranking_full.qini_above_random,
            'uplift_at_10pct': result.uplift_at_k['10pct'],
            'uplift_at_20pct': result.uplift_at_k['20pct'],
            'uplift_at_30pct': result.uplift_at_k['30pct'],
            'incremental_conversions_at_10pct': result.incremental_conversions_at_k['10pct'],
            'incremental_conversions_at_20pct': result.incremental_conversions_at_k['20pct'],
            'incremental_conversions_at_30pct': result.incremental_conversions_at_k['30pct'],
        })
    model_summary_rows_full = list(_full_rows_with_run_context(model_summary_rows_full))

    random_deciles_rows_full = list(_full_rows_with_run_context(random_ranking_full.decile_table.to_dict('records')))
    response_deciles_rows_full = list(_full_rows_with_run_context(response_ranking_full.decile_table.to_dict('records')))

    for name, rows in (
        ('tables/ate_summary.csv', ate_summary_rows_full),
        ('tables/response_diagnostics.csv', response_diagnostics_rows_full),
        ('tables/random_deciles.csv', random_deciles_rows_full),
        ('tables/response_deciles.csv', response_deciles_rows_full),
        ('tables/uplift_at_k.csv', uplift_at_k_rows_full),
        ('tables/model_summary.csv', model_summary_rows_full),
    ):
        write_text_new(FULL_RUN_ROOT, name, pd.DataFrame(rows).to_csv(index=False, lineterminator='\n'))

    print('FULL tables written.')


In [25]:
if RUN_T07_STAGE:
    response_predictions_frame_full = pd.DataFrame({
        SOURCE_ROW_ID: source_row_id_full_validation,
        'response_probability': response_probabilities_full,
    })
    random_scores_frame_full = pd.DataFrame({
        SOURCE_ROW_ID: source_row_id_full_validation,
        'random_score': random_scores_full,
    })
    response_predictions_bytes_full = response_predictions_frame_full.to_parquet(index=False)
    random_scores_bytes_full = random_scores_frame_full.to_parquet(index=False)

    write_bytes_new(FULL_RUN_ROOT, 'predictions/development/response/seed_42/validation_predictions.parquet', response_predictions_bytes_full)
    write_bytes_new(FULL_RUN_ROOT, 'predictions/development/random/seed_42/validation_scores.parquet', random_scores_bytes_full)

    response_model_text_full = response_model_full.booster.model_to_string()
    write_text_new(FULL_RUN_ROOT, 'models/response_model.txt', response_model_text_full)

    random_baseline_summary_rows_full = list(_full_rows_with_run_context([{
        'theoretical_random_qini_area': random_ranking_full.theoretical_random_qini_area,
        'illustrative_draw_seed': metrics.RANDOM_RANKING_SEED,
        'illustrative_draw_qini_area': random_ranking_full.qini_area,
        'illustrative_draw_qini_above_random': random_ranking_full.qini_above_random,
    }]))
    write_text_new(FULL_RUN_ROOT, 'audit/random_baseline_summary.csv', pd.DataFrame(random_baseline_summary_rows_full).to_csv(index=False, lineterminator='\n'))

    random_baseline_draws_rows_full = list(_full_rows_with_run_context(random_reference_distribution_full.to_dict('records')))
    write_text_new(FULL_RUN_ROOT, 'audit/random_baseline_draws.csv', pd.DataFrame(random_baseline_draws_rows_full).to_csv(index=False, lineterminator='\n'))

    model_probability_diagnostics_rows_full = list(_full_rows_with_run_context([{'method': 'response', **response_diag_full.__dict__}]))
    write_text_new(FULL_RUN_ROOT, 'audit/model_probability_diagnostics.csv', pd.DataFrame(model_probability_diagnostics_rows_full).to_csv(index=False, lineterminator='\n'))

    definitions_full = metrics.metric_definitions()
    definitions_sha256_full = hashlib.sha256(json.dumps(definitions_full, sort_keys=True).encode()).hexdigest()
    definitions_rows_full = list(_full_rows_with_run_context(
        [{'field': k, 'value': _flatten(v)} for k, v in definitions_full.items()]
        + [{'field': 'definitions_sha256', 'value': definitions_sha256_full}]
    ))
    write_text_new(FULL_RUN_ROOT, 'audit/metric_definitions.csv', pd.DataFrame(definitions_rows_full).to_csv(index=False, lineterminator='\n'))

    response_predictions_sha256_full = hashlib.sha256(response_predictions_bytes_full).hexdigest()
    random_scores_sha256_full = hashlib.sha256(random_scores_bytes_full).hexdigest()
    print('FULL audit/model/prediction artifacts written.')


In [26]:
if RUN_T07_STAGE:
    import io


    def _cumulative_rate_curve(decile_table):
        ordered = decile_table.sort_values('decile')
        cum_n1 = ordered['n1'].cumsum()
        cum_n0 = ordered['n0'].cumsum()
        cum_y1 = ordered['y1'].cumsum()
        cum_y0 = ordered['y0'].cumsum()
        with np.errstate(divide='ignore', invalid='ignore'):
            cumulative_rate = (cum_y1 / cum_n1) - (cum_y0 / cum_n0)
        return ordered['decile'].to_numpy(), cumulative_rate.to_numpy()


    def _save_figure_bytes(fig, run_root, relative_path):
        buffer = io.BytesIO()
        fig.savefig(buffer, format='png', dpi=120, bbox_inches='tight')
        plt.close(fig)
        write_bytes_new(run_root, relative_path, buffer.getvalue())


    fig, ax = plt.subplots(figsize=(7, 4))
    deciles = response_ranking_full.decile_table.sort_values('decile')['decile']
    ax.bar(deciles - 0.15, response_ranking_full.decile_table.sort_values('decile')['observed_uplift'], width=0.3, label='Response')
    ax.bar(deciles + 0.15, random_ranking_full.decile_table.sort_values('decile')['observed_uplift'], width=0.3, label='Random')
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_xlabel('Decile (1 = highest-ranked)')
    ax.set_ylabel('Observed uplift (treated_rate - control_rate)')
    ax.set_title('Response vs. Random: observed uplift by decile')
    ax.legend()
    fig.tight_layout()
    _save_figure_bytes(fig, FULL_RUN_ROOT, 'figures/response_uplift_deciles.png')

    fig, ax = plt.subplots(figsize=(7, 4))
    resp_d, resp_rate = _cumulative_rate_curve(response_ranking_full.decile_table)
    rand_d, rand_rate = _cumulative_rate_curve(random_ranking_full.decile_table)
    ax.plot(resp_d, resp_rate, marker='o', label='Response')
    ax.plot(rand_d, rand_rate, marker='o', label='Random')
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_xlabel('Cumulative decile coverage')
    ax.set_ylabel('Cumulative uplift rate')
    ax.set_title('Cumulative uplift rate by coverage')
    ax.legend()
    fig.tight_layout()
    _save_figure_bytes(fig, FULL_RUN_ROOT, 'figures/cumulative_uplift_rate.png')

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(response_ranking_full.qini_curve['coverage'], response_ranking_full.qini_curve['qini_gain'], label='Response')
    ax.plot(random_ranking_full.qini_curve['coverage'], random_ranking_full.qini_curve['qini_gain'], label='Random (one draw)')
    ax.set_xlabel('Coverage')
    ax.set_ylabel('Qini gain (incremental conversions)')
    ax.set_title('Cumulative Qini gain by coverage')
    ax.legend()
    fig.tight_layout()
    _save_figure_bytes(fig, FULL_RUN_ROOT, 'figures/cumulative_uplift_gain.png')

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(response_ranking_full.qini_curve['coverage'], response_ranking_full.qini_curve['qini_gain'], label='Response')
    ax.plot(random_ranking_full.qini_curve['coverage'], random_ranking_full.qini_curve['qini_gain'], label='Random (one draw)')
    theoretical_x = np.array([0.0, 1.0])
    theoretical_y = theoretical_x * (response_ranking_full.qini_curve['qini_gain'].iloc[-1])
    ax.plot(theoretical_x, theoretical_y, linestyle='--', color='gray', label='Theoretical random line')
    ax.set_xlabel('Coverage')
    ax.set_ylabel('Qini gain')
    ax.set_title('Qini curve: Response vs. Random vs. theoretical random')
    ax.legend()
    fig.tight_layout()
    _save_figure_bytes(fig, FULL_RUN_ROOT, 'figures/qini_curve.png')

    print('FULL figures written.')


### FULL reproducibility check (T07.7)

In [27]:
if RUN_T07_STAGE:
    reloaded_booster_full = lgb.Booster(model_str=response_model_text_full)
    reloaded_config_hash_full = lgb_baseline.config_hash()
    config_hash_matches_full = reloaded_config_hash_full == response_model_full.config_hash

    X_full_validation_rebuilt = transform_full.transform(full_validation_frame)
    reloaded_probabilities_full = np.asarray(
        reloaded_booster_full.predict(X_full_validation_rebuilt, num_iteration=response_model_full.best_iteration), dtype=np.float64
    )
    prediction_reload_matches_full = np.allclose(reloaded_probabilities_full, response_probabilities_full, rtol=1e-6, atol=1e-8)
    row_identity_matches_full = set(source_row_id_full_validation) == set(validation_ids_full)

    random_scores_regenerated_full = metrics.seeded_random_scores(len(validation_ids_full), seed=metrics.RANDOM_RANKING_SEED)
    random_scores_exact_match_full = np.array_equal(random_scores_regenerated_full, random_scores_full)

    # The frozen 200-draw distribution is not recomputed a second time to "prove" determinism here --
    # T06's own test suite (test_random_ranking_reference_is_deterministic_given_master_seed) already
    # covers that. This is a bounded self-consistency check of what was actually stored above.
    random_reference_self_consistent_full = (
        len(random_reference_distribution_full) == metrics.RANDOM_RANKING_REFERENCE_DRAWS == 200
        and random_reference_distribution_full['draw_index'].nunique() == 200
    )

    reload_verification_full = {
        'config_hash_matches': bool(config_hash_matches_full),
        'prediction_reload_matches_within_tolerance': bool(prediction_reload_matches_full),
        'row_identity_matches': bool(row_identity_matches_full),
        'random_scores_exact_match': bool(random_scores_exact_match_full),
        'random_reference_200_draws_self_consistent': bool(random_reference_self_consistent_full),
        'tolerance': {'rtol': 1e-6, 'atol': 1e-8},
    }
    print(reload_verification_full)
    assert all(reload_verification_full[k] for k in (
        'config_hash_matches', 'prediction_reload_matches_within_tolerance',
        'row_identity_matches', 'random_scores_exact_match', 'random_reference_200_draws_self_consistent',
    ))


In [28]:
if RUN_T07_STAGE:
    full_wall_seconds = __import__('time').perf_counter() - full_wall_start
    full_peak_rss_bytes = process.memory_info().rss

    write_json_new(FULL_RUN_ROOT, 'audit/environment.json', {
        'run_id': FULL_RUN_ID,
        'created_at_utc': full_started.isoformat(),
        'git_head': full_git_head,
        'git_dirty': full_git_dirty,
        'python': sys.version,
        'numpy': np.__version__,
        'pandas': pd.__version__,
        'scikit_learn': sklearn.__version__,
        'lightgbm': lgb.__version__,
        'psutil': psutil.__version__,
        'total_ram_bytes': psutil.virtual_memory().total,
    })

    write_json_new(FULL_RUN_ROOT, 'audit/run_config.json', {
        'run_id': FULL_RUN_ID,
        'created_at_utc': full_started.isoformat(),
        'stage': 't07_full',
        'population': 'full_development_population',
        'git_head': full_git_head,
        'git_dirty': full_git_dirty,
        'real_or_held_out_data_accessed': False,
        'notebook_source_sha256': notebook_source_sha256(NOTEBOOK_PATH) if NOTEBOOK_PATH.is_file() else None,
        'src_lightgbm_baseline_sha256': sha256_file(REPO_ROOT / 'src' / 'lightgbm_baseline.py'),
        'src_metrics_sha256': sha256_file(REPO_ROOT / 'src' / 'metrics.py'),
        'processed_sha256': processed_sha256,
        'split_membership_sha256': observed_membership_hash,
        't05_run_id': t05_config['lifecycle_state_evidence']['authorizing_run_id'],
        't04_lifecycle_state': t04_config['lifecycle_state'],
        't06_authoritative_run_id': t07_config['input']['metrics_interface']['authoritative_t06_run_id'],
        't06_authoritative_definitions_sha256': t07_config['input']['metrics_interface']['authoritative_t06_definitions_sha256'],
        'scale_gating': {'policy': 'D30', 'stage': 'FULL', 'resource_gates': RESOURCE_GATES, 'preceding_smoke_run_id': RUN_ID},
        'train_count': int(len(train_ids_full)),
        'validation_count': int(len(validation_ids_full)),
        'lightgbm_config': lgb_baseline.FROZEN_BINARY_CONFIG,
        'lightgbm_config_hash': response_model_full.config_hash,
        'lightgbm_best_iteration': response_model_full.best_iteration,
        'lightgbm_num_boost_round_cap': lgb_baseline.NUM_BOOST_ROUND_CAP,
        'lightgbm_early_stopping_rounds': lgb_baseline.EARLY_STOPPING_ROUNDS,
        'response_predictions_sha256': response_predictions_sha256_full,
        'random_scores_sha256': random_scores_sha256_full,
        'definitions_sha256': definitions_sha256_full,
        'resource_evidence': {
            'wall_seconds': full_wall_seconds,
            'baseline_rss_bytes': full_baseline_rss_bytes,
            'peak_rss_bytes': full_peak_rss_bytes,
            'peak_rss_delta_bytes': full_peak_rss_bytes - full_baseline_rss_bytes,
        },
        'reload_verification': reload_verification_full,
    })

    print(f'FULL resource evidence: wall_seconds={full_wall_seconds:.2f}, peak_rss_delta_bytes={full_peak_rss_bytes - full_baseline_rss_bytes:,}')


In [29]:
if RUN_T07_STAGE:
    finalize_artifact_manifest(
        FULL_RUN_ROOT,
        run_id=FULL_RUN_ID,
        final_status='COMPLETED_T07_FULL_VERIFIED',
        created_at_utc=datetime.now(timezone.utc).isoformat(),
        stage='t07_full',
        population='full_development_population',
        external_artifacts=[
            {'path': 'kaggle/02_uplift_modeling.ipynb#sources', 'role': 'human_readable_protocol_source',
             'sha256': notebook_source_sha256(NOTEBOOK_PATH) if NOTEBOOK_PATH.is_file() else None, 'status': 'PASS'},
            {'path': 'src/lightgbm_baseline.py', 'role': 'reusable_t07_contract', 'sha256': sha256_file(REPO_ROOT / 'src' / 'lightgbm_baseline.py'), 'status': 'PASS'},
            {'path': 'src/metrics.py', 'role': 'reusable_t06_contract', 'sha256': sha256_file(REPO_ROOT / 'src' / 'metrics.py'), 'status': 'PASS'},
            {'path': f'outputs/runs/{RUN_ID}', 'role': 'preceding_smoke_run', 'sha256': None, 'status': 'PASS'},
        ],
    )
    print(f'FULL run finalized: {FULL_RUN_ID}')

    immutable_write_refused_full = False
    try:
        write_json_new(FULL_RUN_ROOT, 'audit/should_be_refused.json', {'x': 1})
    except (FileExistsError, DataContractError):
        immutable_write_refused_full = True
    except Exception:
        immutable_write_refused_full = False
    print(f'Immutable-run write refusal verified: {immutable_write_refused_full}')
    assert immutable_write_refused_full


In [30]:
if RUN_T07_STAGE:
    full_summary = {
        'run_id': FULL_RUN_ID,
        'preceding_smoke_run_id': RUN_ID,
        'train_count': int(len(train_ids_full)),
        'validation_count': int(len(validation_ids_full)),
        'lightgbm_version': lgb.__version__,
        'lightgbm_config_hash': response_model_full.config_hash,
        'best_iteration': response_model_full.best_iteration,
        'response_diagnostics': response_diag_full.__dict__,
        'random_reference_draws': len(random_reference_distribution_full),
        'random_theoretical_qini_area': random_ranking_full.theoretical_random_qini_area,
        'random_illustrative_qini_above_random': random_ranking_full.qini_above_random,
        'response_qini_area': response_ranking_full.qini_area,
        'response_qini_above_random': response_ranking_full.qini_above_random,
        'response_uplift_at_k': response_ranking_full.uplift_at_k,
        'response_incremental_conversions_at_k': response_ranking_full.incremental_conversions_at_k,
        'ate': ate_full.__dict__,
        'reload_verification': reload_verification_full,
        'resource_wall_seconds': full_wall_seconds,
        'immutable_write_refused': immutable_write_refused_full,
    }
    print(json.dumps(full_summary, indent=2, default=str))


## 6. Interpretation

In [31]:
if RUN_T07_STAGE:
    display(Markdown(
        f"Response's own diagnostics (ROC-AUC {response_diag_full.roc_auc:.4f}) describe how well "
        f"`P(Y=1|X)` predicts conversion -- a factual-outcome quality measure, never a causal ranking claim "
        f"(D12, D27). Whether that translates into a *useful uplift ranking* is a separate question, answered "
        f"only by routing the same scores through the frozen T06 Qini/uplift-at-K interface: Response reaches "
        f"`qini_above_random = {response_ranking_full.qini_above_random:.4f}` against a theoretical random "
        f"reference of {response_ranking_full.theoretical_random_qini_area:.4f}, and the illustrative seeded "
        f"random ranking reaches `qini_above_random = {random_ranking_full.qini_above_random:.4f}` of its own "
        f"-- both are read directly off the same metric interface with no separate formula for either method.\n\n"
        f"{'Response ranks above the random reference on this development population.' if response_ranking_full.qini_above_random > random_ranking_full.qini_above_random else 'Response does not clearly outrank the random reference on this development population -- a strong response model is not automatically a good uplift ranker, and that is a legitimate, reportable finding here, not an error.'} "
        f"This is a development-only observation on the validation partition; it is not a held-out claim, and "
        f"predicted uplift is not a true individual treatment effect. T-Learner, X-Learner, and Causal Forest "
        f"(T08-T11) extend this same comparison with genuinely causal estimators."
    ))


In [32]:
RUN_T08_SMOKE_STAGE = False  # T08 SMOKE is already accepted.
T08_SMOKE_RUN_ID_ACCEPTED = 't08_smoke_20260818T154813Z_381508'
T08_SMOKE_ERRATUM_RUN_ID_ACCEPTED = 't08_smoke_audit_erratum_20260818T155806Z_642086'

if not RUN_T08_SMOKE_STAGE:
    t08_smoke_root = REPO_ROOT / 'outputs' / 'runs' / T08_SMOKE_RUN_ID_ACCEPTED
    t08_smoke_manifest_path = t08_smoke_root / 'audit' / 'artifact_manifest.json'
    if not t08_smoke_manifest_path.is_file():
        raise RuntimeError(
            f'RUN_T08_SMOKE_STAGE is False, but the accepted T08 SMOKE run evidence was not found at '
            f'{t08_smoke_root}. Refusing to silently set RUN_T08_SMOKE_STAGE = True and refit the SMOKE '
            f'mu1/mu0 models -- either provide/mount the accepted governed run evidence '
            f'(outputs/runs/{T08_SMOKE_RUN_ID_ACCEPTED}/), or explicitly set RUN_T08_SMOKE_STAGE = True '
            f'in this cell to opt into reproducing T08 SMOKE from scratch.'
        )
    t08_smoke_manifest = json.loads(t08_smoke_manifest_path.read_text(encoding='utf-8'))
    t08_smoke_artifact_hashes = {a['path']: a['sha256'] for a in t08_smoke_manifest['artifacts']}
    t08_smoke_model_summary_path = t08_smoke_root / 'tables' / 'model_summary.csv'
    t08_smoke_model_summary_actual_sha256 = hashlib.sha256(t08_smoke_model_summary_path.read_bytes()).hexdigest()
    t08_smoke_model_summary_expected_sha256 = t08_smoke_artifact_hashes.get('tables/model_summary.csv')
    if t08_smoke_model_summary_expected_sha256 is None or t08_smoke_model_summary_actual_sha256 != t08_smoke_model_summary_expected_sha256:
        raise RuntimeError(
            f'T08 SMOKE tables/model_summary.csv hash does not match its own artifact manifest '
            f'(expected {t08_smoke_model_summary_expected_sha256}, actual {t08_smoke_model_summary_actual_sha256}). '
            f'Refusing to reuse unverified evidence.'
        )
    print('Using hash-verified frozen development results (T08 SMOKE).')
    print(f'  Reproducibility: T08 SMOKE run_id={T08_SMOKE_RUN_ID_ACCEPTED}')
else:
    print('RUN_T08_SMOKE_STAGE = True: T08 SMOKE will be recomputed from scratch below.')


Using hash-verified frozen development results (T08 SMOKE).
  Reproducibility: T08 SMOKE run_id=t08_smoke_20260818T154813Z_381508


#### Reproducibility note (technical): T08 SMOKE verification

T08 (T-Learner) uses the D30 path declared in `configs/t08_tlearner.json`:
`SMOKE -> FULL`, `resource_gates = 0`, `smoke_size = 200,000`. This is larger
than T07's SMOKE (50,000) because T-Learner's two arm-specific fits each see
only their own arm's slice of any draw -- the binding constraint is the
control arm's converter cell (`T=0,Y=1`), which is far rarer, per-arm, than
what a single pooled model like T07's Response needed. SMOKE here proves
correctness and artifact mechanics only, for **both** surfaces; it is not a
performance estimate, and this stage's SMOKE population is deliberately not
used for any performance claim.


*(This section -- T08 SMOKE -- is already accepted. It is skipped by default on Run All; see `RUN_T08_SMOKE_STAGE` immediately above. Set it to `True` only to deliberately reproduce T08 SMOKE from scratch.)*

In [33]:
if RUN_T08_SMOKE_STAGE:
    from src.tlearner import (
        TLearnerContractError, assert_aligned_predictions, compute_tau,
        partition_by_arm, reconcile_reloaded_tau,
    )
    import src.tlearner as tlearner_module

    t08_config = json.loads((REPO_ROOT / 'configs' / 't08_tlearner.json').read_text(encoding='utf-8'))

    SMOKE_SIZE_T08 = t08_config['scale_gating']['smoke_size']
    SMOKE_SEED_T08 = t08_config['scale_gating']['smoke_seed']
    RESOURCE_GATES_T08 = t08_config['scale_gating']['resource_gates']

    t08_smoke_started = datetime.now(timezone.utc)
    t08_smoke_wall_start = __import__('time').perf_counter()
    t08_smoke_baseline_rss_bytes = process.memory_info().rss
    t08_smoke_baseline_peak_wset = getattr(process.memory_info(), 'peak_wset', None)

    RUN_ID_T08_SMOKE = t08_smoke_started.strftime('t08_smoke_%Y%m%dT%H%M%SZ_%f')
    RUN_ROOT_T08_SMOKE = REPO_ROOT / 'outputs' / 'runs' / RUN_ID_T08_SMOKE
    RUN_ROOT_T08_SMOKE.mkdir(parents=True, exist_ok=False)

    try:
        t08_git_head = subprocess.run(['git', 'rev-parse', 'HEAD'], cwd=REPO_ROOT, check=True, capture_output=True, text=True).stdout.strip()
        t08_git_dirty = bool(subprocess.run(['git', 'status', '--porcelain'], cwd=REPO_ROOT, check=True, capture_output=True, text=True).stdout.strip())
    except (OSError, subprocess.CalledProcessError):
        t08_git_head, t08_git_dirty = None, None

    print(f'RUN_ID_T08_SMOKE = {RUN_ID_T08_SMOKE}')
    print(f'resource_gates = {RESOURCE_GATES_T08}, smoke_size = {SMOKE_SIZE_T08}, seed = {SMOKE_SEED_T08}')


In [34]:
if RUN_T08_SMOKE_STAGE:
    def _joint_strata_t08(frame, treatment_column, outcome_column):
        return frame[treatment_column].astype(str) + '_' + frame[outcome_column].astype(str)


    def smoke_sample_partition_t08(partition_ids, quota, full_frame, seed):
        subset = full_frame.loc[full_frame[SOURCE_ROW_ID].isin(partition_ids)]
        strata = _joint_strata_t08(subset, TREATMENT_COLUMN, PRIMARY_OUTCOME)
        selected_ids, _ = train_test_split(
            subset[SOURCE_ROW_ID].to_numpy(), train_size=quota, random_state=seed, stratify=strata,
        )
        return np.sort(selected_ids)


    p_train_t08 = len(train_ids_full) / (len(train_ids_full) + len(validation_ids_full))
    train_quota_t08 = round(SMOKE_SIZE_T08 * p_train_t08)
    validation_quota_t08 = SMOKE_SIZE_T08 - train_quota_t08

    smoke_train_ids_t08 = smoke_sample_partition_t08(train_ids_full, train_quota_t08, full_frame, SMOKE_SEED_T08)
    smoke_validation_ids_t08 = smoke_sample_partition_t08(validation_ids_full, validation_quota_t08, full_frame, SMOKE_SEED_T08)

    smoke_total_t08 = len(smoke_train_ids_t08) + len(smoke_validation_ids_t08)
    assert smoke_total_t08 == SMOKE_SIZE_T08, f'T08 SMOKE total {smoke_total_t08} != {SMOKE_SIZE_T08}'
    assert set(smoke_train_ids_t08).issubset(set(train_ids_full)), 'smoke_train_ids_t08 must be a subset of the frozen train partition'
    assert set(smoke_validation_ids_t08).issubset(set(validation_ids_full)), 'smoke_validation_ids_t08 must be a subset of the frozen validation partition'
    assert set(smoke_train_ids_t08).isdisjoint(set(smoke_validation_ids_t08))
    development_ids_full_t08 = set(train_ids_full) | set(validation_ids_full)
    assert (set(smoke_train_ids_t08) | set(smoke_validation_ids_t08)).issubset(development_ids_full_t08), (
        'every T08 SMOKE-selected ID must belong to train_ids_full union validation_ids_full'
    )
    # Held-out isolation proved by construction only, same as T07's corrected SMOKE -- never read
    # via split_membership.csv's held_out label or SplitDataset.held_out_ids().

    print(f'T08 SMOKE: train={len(smoke_train_ids_t08):,} validation={len(smoke_validation_ids_t08):,} total={smoke_total_t08:,}')
    print('Held-out isolation: proved by construction from train_ids_full/validation_ids_full only; held-out never read.')


In [35]:
if RUN_T08_SMOKE_STAGE:
    def joint_ty_support_t08(ids, full_frame):
        subset = full_frame.loc[full_frame[SOURCE_ROW_ID].isin(ids)]
        counts = subset.groupby([TREATMENT_COLUMN, PRIMARY_OUTCOME], observed=True).size()
        counts = counts.reindex(pd.MultiIndex.from_product([[0, 1], [0, 1]], names=[TREATMENT_COLUMN, PRIMARY_OUTCOME]), fill_value=0)
        return {f'T={t},Y={y}': int(n) for (t, y), n in counts.items()}


    smoke_train_support_t08 = joint_ty_support_t08(smoke_train_ids_t08, full_frame)
    smoke_validation_support_t08 = joint_ty_support_t08(smoke_validation_ids_t08, full_frame)
    print('T08 SMOKE train (T,Y) support:', smoke_train_support_t08)
    print('T08 SMOKE validation (T,Y) support:', smoke_validation_support_t08)

    t08_smoke_all_cells_nonempty = all(n > 0 for n in {**smoke_train_support_t08, **smoke_validation_support_t08}.values())
    print('All 8 partition x (T,Y) cells non-empty:', t08_smoke_all_cells_nonempty)
    if not t08_smoke_all_cells_nonempty:
        raise TLearnerContractError(
            'T08 SMOKE support is degenerate: at least one train/validation x (T,Y) cell is empty at '
            f'smoke_size={SMOKE_SIZE_T08}. SMOKE purpose is engineering/correctness validation, not '
            'performance estimation -- failing closed rather than adjusting the sample after seeing this.'
        )


In [36]:
if RUN_T08_SMOKE_STAGE:
    smoke_sample_ids_frame_t08 = pd.concat([
        pd.DataFrame({SOURCE_ROW_ID: smoke_train_ids_t08, 'partition': 'train'}),
        pd.DataFrame({SOURCE_ROW_ID: smoke_validation_ids_t08, 'partition': 'validation'}),
    ], ignore_index=True)
    smoke_sample_ids_sha256_t08 = hashlib.sha256(
        smoke_sample_ids_frame_t08.sort_values(SOURCE_ROW_ID)[SOURCE_ROW_ID].to_numpy(dtype='<i8').tobytes()
    ).hexdigest()

    write_bytes_new(RUN_ROOT_T08_SMOKE, 'audit/smoke_sample_row_ids.parquet', smoke_sample_ids_frame_t08.to_parquet(index=False))
    write_json_new(RUN_ROOT_T08_SMOKE, 'audit/smoke_sample_manifest.json', {
        'run_id': RUN_ID_T08_SMOKE,
        'stage': 't08_smoke',
        'population': 'smoke_200000_rows',
        'smoke_size': SMOKE_SIZE_T08,
        'smoke_seed': SMOKE_SEED_T08,
        'train_quota': int(train_quota_t08),
        'validation_quota': int(validation_quota_t08),
        'train_count': int(len(smoke_train_ids_t08)),
        'validation_count': int(len(smoke_validation_ids_t08)),
        'total_count': int(smoke_total_t08),
        'train_support': smoke_train_support_t08,
        'validation_support': smoke_validation_support_t08,
        'all_partition_ty_cells_nonempty': bool(t08_smoke_all_cells_nonempty),
        'smoke_sample_row_ids_sha256': smoke_sample_ids_sha256_t08,
        'held_out_isolation_method': (
            'guaranteed_by_construction_from_sanctioned_development_partitions: smoke_train_ids_t08 and '
            'smoke_validation_ids_t08 are proved subsets of train_ids_full/validation_ids_full (each obtained '
            'solely via SplitDataset.train_ids()/.validation_ids()), pairwise disjoint, and their union is a '
            'subset of train_ids_full union validation_ids_full -- held-out is never read via '
            'split_membership.csv\'s held_out label, SplitDataset.held_out_ids(), or any other path'
        ),
    })
    print('T08 SMOKE sample identity persisted.')


In [37]:
if RUN_T08_SMOKE_STAGE:
    smoke_train_frame_t08 = full_frame.loc[full_frame[SOURCE_ROW_ID].isin(smoke_train_ids_t08)].sort_values(SOURCE_ROW_ID).reset_index(drop=True)
    smoke_validation_frame_t08 = full_frame.loc[full_frame[SOURCE_ROW_ID].isin(smoke_validation_ids_t08)].sort_values(SOURCE_ROW_ID).reset_index(drop=True)

    transform_t08_smoke = IdentityFeatureTransform()
    X_smoke_train_t08 = transform_t08_smoke.fit_transform(smoke_train_frame_t08)
    X_smoke_validation_t08 = transform_t08_smoke.transform(smoke_validation_frame_t08)
    assert_model_feature_contract(X_smoke_train_t08.columns)
    assert_model_feature_contract(X_smoke_validation_t08.columns)

    y_smoke_train_t08 = smoke_train_frame_t08[PRIMARY_OUTCOME].astype('float64')
    y_smoke_validation_t08 = smoke_validation_frame_t08[PRIMARY_OUTCOME].astype('float64')
    t_smoke_train_t08 = smoke_train_frame_t08[TREATMENT_COLUMN].astype('float64').to_numpy()
    t_smoke_validation_t08 = smoke_validation_frame_t08[TREATMENT_COLUMN].astype('float64').to_numpy()
    y_smoke_validation_arr_t08 = y_smoke_validation_t08.to_numpy()
    source_row_id_smoke_validation_t08 = smoke_validation_frame_t08[SOURCE_ROW_ID].to_numpy()

    print(f'X_smoke_train_t08: {X_smoke_train_t08.shape}, X_smoke_validation_t08: {X_smoke_validation_t08.shape}')
    print('Feature contract verified: X is exactly', tuple(X_smoke_train_t08.columns))


### T08.1 Arm partitioning

In [38]:
if RUN_T08_SMOKE_STAGE:
    (X_train_treated_t08,), (X_train_control_t08,) = partition_by_arm(t_smoke_train_t08, X_smoke_train_t08.to_numpy())
    X_train_treated_t08 = pd.DataFrame(X_train_treated_t08, columns=X_smoke_train_t08.columns)
    X_train_control_t08 = pd.DataFrame(X_train_control_t08, columns=X_smoke_train_t08.columns)
    (y_train_treated_t08,), (y_train_control_t08,) = partition_by_arm(t_smoke_train_t08, y_smoke_train_t08.to_numpy())

    (X_val_treated_t08,), (X_val_control_t08,) = partition_by_arm(t_smoke_validation_t08, X_smoke_validation_t08.to_numpy())
    X_val_treated_t08 = pd.DataFrame(X_val_treated_t08, columns=X_smoke_validation_t08.columns)
    X_val_control_t08 = pd.DataFrame(X_val_control_t08, columns=X_smoke_validation_t08.columns)
    (y_val_treated_t08,), (y_val_control_t08,) = partition_by_arm(t_smoke_validation_t08, y_smoke_validation_arr_t08)

    assert len(X_train_treated_t08) == smoke_train_support_t08['T=1,Y=0'] + smoke_train_support_t08['T=1,Y=1']
    assert len(X_train_control_t08) == smoke_train_support_t08['T=0,Y=0'] + smoke_train_support_t08['T=0,Y=1']
    print(f'mu1 train rows: {len(X_train_treated_t08):,} (arm-specific early-stopping validation: {len(X_val_treated_t08):,})')
    print(f'mu0 train rows: {len(X_train_control_t08):,} (arm-specific early-stopping validation: {len(X_val_control_t08):,})')


### T08.2 Fit mu1 and mu0 (identical frozen config)

In [39]:
if RUN_T08_SMOKE_STAGE:
    mu1_model_smoke = lgb_baseline.fit_binary_classifier(
        X_train_treated_t08, y_train_treated_t08, X_val_treated_t08, y_val_treated_t08,
    )
    mu0_model_smoke = lgb_baseline.fit_binary_classifier(
        X_train_control_t08, y_train_control_t08, X_val_control_t08, y_val_control_t08,
    )
    assert mu1_model_smoke.config_hash == mu0_model_smoke.config_hash, 'mu1/mu0 must share the identical frozen config'
    print(f'mu1: best_iteration={mu1_model_smoke.best_iteration}, config_hash={mu1_model_smoke.config_hash[:16]}...')
    print(f'mu0: best_iteration={mu0_model_smoke.best_iteration}, config_hash={mu0_model_smoke.config_hash[:16]}...')


### T08.3 Score the full common SMOKE validation cohort

In [40]:
if RUN_T08_SMOKE_STAGE:
    mu1_hat_smoke = lgb_baseline.predict_probabilities(mu1_model_smoke, X_smoke_validation_t08)
    mu0_hat_smoke = lgb_baseline.predict_probabilities(mu0_model_smoke, X_smoke_validation_t08)

    assert_aligned_predictions(source_row_id_smoke_validation_t08, source_row_id_smoke_validation_t08, source_row_id_smoke_validation_t08)
    assert len(mu1_hat_smoke) == len(smoke_validation_ids_t08) == len(mu0_hat_smoke)
    print('Both surfaces scored the full SMOKE validation cohort:', len(mu1_hat_smoke), 'rows')


### T08.4 tau_hat = mu1_hat - mu0_hat

In [41]:
if RUN_T08_SMOKE_STAGE:
    tau_hat_smoke = compute_tau(mu1_hat_smoke, mu0_hat_smoke)
    np.testing.assert_array_equal(tau_hat_smoke, mu1_hat_smoke - mu0_hat_smoke)  # exact, elementwise, asserted not just constructed

    positive_count = int((tau_hat_smoke > 0).sum())
    negative_count = int((tau_hat_smoke < 0).sum())
    zero_count = int((tau_hat_smoke == 0).sum())
    assert positive_count == int(((mu1_hat_smoke - mu0_hat_smoke) > 0).sum())  # sign orientation: higher mu1 -> positive tau
    print(f'tau_hat: positive={positive_count:,} negative={negative_count:,} zero={zero_count:,}')


### T08.5 Evaluate via T06 only

In [42]:
if RUN_T08_SMOKE_STAGE:
    tlearner_ranking_smoke = metrics.evaluate_ranking(
        tau_hat_smoke, t_smoke_validation_t08, y_smoke_validation_arr_t08, source_row_id_smoke_validation_t08,
    )
    print(f'T-Learner SMOKE qini_above_random (non-substantive): {tlearner_ranking_smoke.qini_above_random:.6f}')
    # metrics.random_ranking_reference_distribution() and metrics.seeded_random_scores() are never called
    # here -- T08 does not recompute T07's random reference or Response diagnostics.


### T08.6 SMOKE artifacts

In [43]:
if RUN_T08_SMOKE_STAGE:
    def _rows_with_run_context_t08(rows, population='smoke_200000_rows'):
        for row in rows:
            row = dict(row)
            row.setdefault('run_id', RUN_ID_T08_SMOKE)
            row.setdefault('stage', 't08_smoke')
            row.setdefault('population', population)
            yield row


    uplift_at_k_rows_t08 = list(_rows_with_run_context_t08([
        {'method': 'tlearner', 'k': label, 'uplift': tlearner_ranking_smoke.uplift_at_k[label],
         'incremental_conversions': tlearner_ranking_smoke.incremental_conversions_at_k[label],
         'status': tlearner_ranking_smoke.top_k_status[label]}
        for label in metrics.RANKING_K_LABELS
    ]))
    model_summary_rows_t08 = list(_rows_with_run_context_t08([{
        'ranking_method': 'tlearner',
        'qini_area': tlearner_ranking_smoke.qini_area,
        'theoretical_random_qini_area': tlearner_ranking_smoke.theoretical_random_qini_area,
        'qini_above_random': tlearner_ranking_smoke.qini_above_random,
        'uplift_at_10pct': tlearner_ranking_smoke.uplift_at_k['10pct'],
        'uplift_at_20pct': tlearner_ranking_smoke.uplift_at_k['20pct'],
        'uplift_at_30pct': tlearner_ranking_smoke.uplift_at_k['30pct'],
        'incremental_conversions_at_10pct': tlearner_ranking_smoke.incremental_conversions_at_k['10pct'],
        'incremental_conversions_at_20pct': tlearner_ranking_smoke.incremental_conversions_at_k['20pct'],
        'incremental_conversions_at_30pct': tlearner_ranking_smoke.incremental_conversions_at_k['30pct'],
    }]))
    tlearner_deciles_rows_t08 = list(_rows_with_run_context_t08(tlearner_ranking_smoke.decile_table.to_dict('records')))

    for name, rows in (
        ('tables/uplift_at_k.csv', uplift_at_k_rows_t08),
        ('tables/model_summary.csv', model_summary_rows_t08),
        ('tables/tlearner_deciles.csv', tlearner_deciles_rows_t08),
    ):
        write_text_new(RUN_ROOT_T08_SMOKE, name, pd.DataFrame(rows).to_csv(index=False, lineterminator='\n'))

    tlearner_predictions_frame_smoke = pd.DataFrame({
        SOURCE_ROW_ID: source_row_id_smoke_validation_t08,
        'mu1_hat': mu1_hat_smoke,
        'mu0_hat': mu0_hat_smoke,
        'tau_hat': tau_hat_smoke,
    })
    tlearner_predictions_bytes_smoke = tlearner_predictions_frame_smoke.to_parquet(index=False)
    write_bytes_new(RUN_ROOT_T08_SMOKE, 'predictions/development/tlearner/seed_42/validation_predictions.parquet', tlearner_predictions_bytes_smoke)

    mu1_model_text_smoke = mu1_model_smoke.booster.model_to_string()
    mu0_model_text_smoke = mu0_model_smoke.booster.model_to_string()
    write_text_new(RUN_ROOT_T08_SMOKE, 'models/tlearner_mu1.txt', mu1_model_text_smoke)
    write_text_new(RUN_ROOT_T08_SMOKE, 'models/tlearner_mu0.txt', mu0_model_text_smoke)

    def _flatten_t08(value):
        return value if isinstance(value, (str, int, float, bool)) or value is None else json.dumps(value)


    definitions_t08 = metrics.metric_definitions()
    definitions_sha256_t08 = hashlib.sha256(json.dumps(definitions_t08, sort_keys=True).encode()).hexdigest()
    definitions_rows_t08 = list(_rows_with_run_context_t08(
        [{'field': k, 'value': _flatten_t08(v)} for k, v in definitions_t08.items()]
        + [{'field': 'definitions_sha256', 'value': definitions_sha256_t08}]
    ))
    write_text_new(RUN_ROOT_T08_SMOKE, 'audit/metric_definitions.csv', pd.DataFrame(definitions_rows_t08).to_csv(index=False, lineterminator='\n'))

    tlearner_predictions_sha256_smoke = hashlib.sha256(tlearner_predictions_bytes_smoke).hexdigest()
    print('T08 SMOKE tables/models/predictions/audit written.')


### T08.7 Reload/reproducibility (derived tau tolerance)

In [44]:
if RUN_T08_SMOKE_STAGE:
    reloaded_mu1_booster_smoke = lgb.Booster(model_str=mu1_model_text_smoke)
    reloaded_mu0_booster_smoke = lgb.Booster(model_str=mu0_model_text_smoke)
    reloaded_config_hash_mu1_smoke = lgb_baseline.config_hash()
    reloaded_config_hash_mu0_smoke = lgb_baseline.config_hash()
    config_hash_matches_smoke = (
        reloaded_config_hash_mu1_smoke == mu1_model_smoke.config_hash
        and reloaded_config_hash_mu0_smoke == mu0_model_smoke.config_hash
    )

    X_smoke_validation_rebuilt_t08 = transform_t08_smoke.transform(smoke_validation_frame_t08)
    mu1_reloaded_smoke = np.asarray(reloaded_mu1_booster_smoke.predict(X_smoke_validation_rebuilt_t08, num_iteration=mu1_model_smoke.best_iteration), dtype=np.float64)
    mu0_reloaded_smoke = np.asarray(reloaded_mu0_booster_smoke.predict(X_smoke_validation_rebuilt_t08, num_iteration=mu0_model_smoke.best_iteration), dtype=np.float64)

    mu1_reload_matches_smoke = bool(np.allclose(mu1_reloaded_smoke, mu1_hat_smoke, rtol=1e-6, atol=1e-8))
    mu0_reload_matches_smoke = bool(np.allclose(mu0_reloaded_smoke, mu0_hat_smoke, rtol=1e-6, atol=1e-8))

    tau_reload_matches_smoke, tau_reloaded_smoke, tau_reload_info_smoke = reconcile_reloaded_tau(
        mu1_reloaded_smoke, mu0_reloaded_smoke, tau_hat_smoke, mu1_hat_smoke, mu0_hat_smoke,
    )

    row_identity_matches_smoke = set(source_row_id_smoke_validation_t08) == set(smoke_validation_ids_t08)

    reload_verification_t08_smoke = {
        'config_hash_matches': bool(config_hash_matches_smoke),
        'mu1_reload_matches_within_tolerance': mu1_reload_matches_smoke,
        'mu0_reload_matches_within_tolerance': mu0_reload_matches_smoke,
        'tau_reload_matches_within_derived_tolerance': bool(tau_reload_matches_smoke),
        'tau_reload_info': tau_reload_info_smoke,
        'row_identity_matches': bool(row_identity_matches_smoke),
        'tolerance': {'rtol': 1e-6, 'atol': 1e-8, 'tau_derived_atol': 2e-8},
    }
    print(reload_verification_t08_smoke)
    assert all([
        reload_verification_t08_smoke['config_hash_matches'],
        reload_verification_t08_smoke['mu1_reload_matches_within_tolerance'],
        reload_verification_t08_smoke['mu0_reload_matches_within_tolerance'],
        reload_verification_t08_smoke['tau_reload_matches_within_derived_tolerance'],
        reload_verification_t08_smoke['row_identity_matches'],
    ])


### T08.8 T07 reuse verification (no recomputation)

In [45]:
if RUN_T08_SMOKE_STAGE:
    t08_t07_reuse_hashes = {}
    for rel_path in ('models/response_model.txt',
                      'predictions/development/response/seed_42/validation_predictions.parquet',
                      'predictions/development/random/seed_42/validation_scores.parquet'):
        actual = hashlib.sha256((t07_full_root / rel_path).read_bytes()).hexdigest()
        expected = t07_artifact_hashes[rel_path]
        t08_t07_reuse_hashes[rel_path] = {'expected': expected, 'actual': actual, 'matches': actual == expected}
        assert actual == expected, f'T07 artifact {rel_path} hash mismatch -- refusing to cite unverified evidence'

    print('T07 reuse hash verification (all must match, none recomputed):')
    for path, info in t08_t07_reuse_hashes.items():
        print('  ' + path + ': matches=' + str(info['matches']))


In [46]:
if RUN_T08_SMOKE_STAGE:
    t08_smoke_wall_seconds = __import__('time').perf_counter() - t08_smoke_wall_start
    t08_smoke_end_memory = process.memory_info()
    t08_smoke_end_peak_wset = getattr(t08_smoke_end_memory, 'peak_wset', None)

    if t08_smoke_baseline_peak_wset is not None and t08_smoke_end_peak_wset is not None:
        # peak_wset is an OS-tracked historical maximum (monotonically non-decreasing since process
        # start), not a before/after instantaneous sample -- valid because T07's own heavy computation
        # is skipped in this run (RUN_T07_STAGE=False), so no prior heavy stage contaminates this figure.
        t08_smoke_resource_evidence = {
            'wall_seconds': t08_smoke_wall_seconds,
            'execution_completed': True,
            'oom_or_termination_observed': False,
            'baseline_peak_wset_bytes': int(t08_smoke_baseline_peak_wset),
            'end_peak_wset_bytes': int(t08_smoke_end_peak_wset),
            'peak_rss_delta_bytes': int(t08_smoke_end_peak_wset) - int(t08_smoke_baseline_peak_wset),
            'measurement_mechanism': 'os_tracked_peak_working_set_since_process_start',
        }
    else:
        t08_smoke_resource_evidence = {
            'wall_seconds': t08_smoke_wall_seconds,
            'execution_completed': True,
            'oom_or_termination_observed': False,
            'peak_rss_delta_bytes': 'NOT_COMPARABLE',
            'peak_rss_delta_not_comparable_reason': 'psutil peak_wset unavailable in this environment; no valid stage-scoped peak measurement mechanism was available.',
        }
    print(t08_smoke_resource_evidence)


In [47]:
if RUN_T08_SMOKE_STAGE:
    write_json_new(RUN_ROOT_T08_SMOKE, 'audit/environment.json', {
        'run_id': RUN_ID_T08_SMOKE,
        'created_at_utc': t08_smoke_started.isoformat(),
        'git_head': t08_git_head,
        'git_dirty': t08_git_dirty,
        'python': sys.version,
        'numpy': np.__version__,
        'pandas': pd.__version__,
        'scikit_learn': sklearn.__version__,
        'lightgbm': lgb.__version__,
        'psutil': psutil.__version__,
        'total_ram_bytes': psutil.virtual_memory().total,
    })

    write_json_new(RUN_ROOT_T08_SMOKE, 'audit/run_config.json', {
        'run_id': RUN_ID_T08_SMOKE,
        'created_at_utc': t08_smoke_started.isoformat(),
        'stage': 't08_smoke',
        'population': 'smoke_200000_rows',
        'git_head': t08_git_head,
        'git_dirty': t08_git_dirty,
        'development_data_accessed': True,
        'held_out_data_accessed': False,
        'notebook_source_sha256': notebook_source_sha256(NOTEBOOK_PATH) if NOTEBOOK_PATH.is_file() else None,
        'src_lightgbm_baseline_sha256': sha256_file(REPO_ROOT / 'src' / 'lightgbm_baseline.py'),
        'src_tlearner_sha256': sha256_file(REPO_ROOT / 'src' / 'tlearner.py'),
        'src_metrics_sha256': sha256_file(REPO_ROOT / 'src' / 'metrics.py'),
        'processed_sha256': processed_sha256,
        'split_membership_sha256': observed_membership_hash,
        't05_run_id': t05_config['lifecycle_state_evidence']['authorizing_run_id'],
        't04_lifecycle_state': t04_config['lifecycle_state'],
        't06_authoritative_run_id': t08_config['input']['metrics_interface']['authoritative_t06_run_id'],
        't06_authoritative_definitions_sha256': t08_config['input']['metrics_interface']['authoritative_t06_definitions_sha256'],
        't07_reuse': {
            'source_run_id': T07_FULL_RUN_ID_ACCEPTED,
            'source_erratum_run_id': T07_ERRATUM_RUN_ID_ACCEPTED,
            'hash_verification': t08_t07_reuse_hashes,
            'recomputed': False,
        },
        'scale_gating': {'policy': 'D30', 'stage': 'SMOKE', 'resource_gates': RESOURCE_GATES_T08, 'smoke_size': SMOKE_SIZE_T08, 'smoke_seed': SMOKE_SEED_T08},
        'lightgbm_config': lgb_baseline.FROZEN_BINARY_CONFIG,
        'mu1_config_hash': mu1_model_smoke.config_hash,
        'mu0_config_hash': mu0_model_smoke.config_hash,
        'mu1_best_iteration': mu1_model_smoke.best_iteration,
        'mu0_best_iteration': mu0_model_smoke.best_iteration,
        'lightgbm_num_boost_round_cap': lgb_baseline.NUM_BOOST_ROUND_CAP,
        'lightgbm_early_stopping_rounds': lgb_baseline.EARLY_STOPPING_ROUNDS,
        'tlearner_predictions_sha256': tlearner_predictions_sha256_smoke,
        'definitions_sha256': definitions_sha256_t08,
        'resource_evidence': t08_smoke_resource_evidence,
        'reload_verification': reload_verification_t08_smoke,
    })
    print('T08 SMOKE run_config.json / environment.json written.')


In [48]:
if RUN_T08_SMOKE_STAGE:
    finalize_artifact_manifest(
        RUN_ROOT_T08_SMOKE,
        run_id=RUN_ID_T08_SMOKE,
        final_status='COMPLETED_T08_SMOKE_VERIFIED',
        created_at_utc=datetime.now(timezone.utc).isoformat(),
        stage='t08_smoke',
        population='smoke_200000_rows',
        external_artifacts=[
            {'path': 'kaggle/02_uplift_modeling.ipynb#sources', 'role': 'human_readable_protocol_source',
             'sha256': notebook_source_sha256(NOTEBOOK_PATH) if NOTEBOOK_PATH.is_file() else None, 'status': 'PASS'},
            {'path': 'src/tlearner.py', 'role': 'reusable_t08_contract', 'sha256': sha256_file(REPO_ROOT / 'src' / 'tlearner.py'), 'status': 'PASS'},
            {'path': 'src/lightgbm_baseline.py', 'role': 'reusable_t07_t08_contract', 'sha256': sha256_file(REPO_ROOT / 'src' / 'lightgbm_baseline.py'), 'status': 'PASS'},
            {'path': 'src/metrics.py', 'role': 'reusable_t06_contract', 'sha256': sha256_file(REPO_ROOT / 'src' / 'metrics.py'), 'status': 'PASS'},
            {'path': f'outputs/runs/{T07_FULL_RUN_ID_ACCEPTED}', 'role': 'reused_t07_evidence', 'sha256': None, 'status': 'PASS'},
        ],
    )
    print(f'T08 SMOKE run finalized: {RUN_ID_T08_SMOKE}')

    immutable_write_refused_t08_smoke = False
    try:
        write_json_new(RUN_ROOT_T08_SMOKE, 'audit/should_be_refused.json', {'x': 1})
    except (FileExistsError, DataContractError):
        immutable_write_refused_t08_smoke = True
    except Exception:
        immutable_write_refused_t08_smoke = False
    print(f'Immutable-run write refusal verified: {immutable_write_refused_t08_smoke}')
    assert immutable_write_refused_t08_smoke


In [49]:
if RUN_T08_SMOKE_STAGE:
    t08_smoke_summary = {
        'run_id': RUN_ID_T08_SMOKE,
        'smoke_total': int(smoke_total_t08),
        'smoke_train_count': int(len(smoke_train_ids_t08)),
        'smoke_validation_count': int(len(smoke_validation_ids_t08)),
        'all_partition_ty_cells_nonempty': bool(t08_smoke_all_cells_nonempty),
        'train_support': smoke_train_support_t08,
        'validation_support': smoke_validation_support_t08,
        'lightgbm_version': lgb.__version__,
        'mu1_config_hash': mu1_model_smoke.config_hash,
        'mu0_config_hash': mu0_model_smoke.config_hash,
        'mu1_best_iteration': mu1_model_smoke.best_iteration,
        'mu0_best_iteration': mu0_model_smoke.best_iteration,
        'tau_sign_counts': {'positive': positive_count, 'negative': negative_count, 'zero': zero_count},
        'tlearner_ranking_smoke_non_substantive': {
            'qini_area': tlearner_ranking_smoke.qini_area,
            'qini_above_random': tlearner_ranking_smoke.qini_above_random,
        },
        'reload_verification': reload_verification_t08_smoke,
        't07_reuse_hash_verification': t08_t07_reuse_hashes,
        'resource_evidence': t08_smoke_resource_evidence,
        'immutable_write_refused': immutable_write_refused_t08_smoke,
    }
    print(json.dumps(t08_smoke_summary, indent=2, default=str))


In [50]:
RUN_T08_FULL_STAGE = False  # T08 FULL is already accepted.
T08_FULL_RUN_ID_ACCEPTED = 't08_full_20260818T160823Z_081282'

if not RUN_T08_FULL_STAGE:
    t08_full_root_ref = REPO_ROOT / 'outputs' / 'runs' / T08_FULL_RUN_ID_ACCEPTED
    t08_full_manifest_path_ref = t08_full_root_ref / 'audit' / 'artifact_manifest.json'
    if not t08_full_manifest_path_ref.is_file():
        raise RuntimeError(
            f'RUN_T08_FULL_STAGE is False, but the accepted T08 FULL run evidence was not found at '
            f'{t08_full_root_ref}. Refusing to silently set RUN_T08_FULL_STAGE = True and refit mu1/mu0 -- '
            f'either provide/mount the accepted governed run evidence '
            f'(outputs/runs/{T08_FULL_RUN_ID_ACCEPTED}/), or explicitly set RUN_T08_FULL_STAGE = True '
            f'in this cell to opt into reproducing T08 FULL from scratch.'
        )
    t08_full_manifest_ref = json.loads(t08_full_manifest_path_ref.read_text(encoding='utf-8'))
    t08_full_artifact_hashes_ref = {a['path']: a['sha256'] for a in t08_full_manifest_ref['artifacts']}
    t08_full_model_summary_path_ref = t08_full_root_ref / 'tables' / 'model_summary.csv'
    t08_full_model_summary_actual_sha256 = hashlib.sha256(t08_full_model_summary_path_ref.read_bytes()).hexdigest()
    t08_full_model_summary_expected_sha256 = t08_full_artifact_hashes_ref.get('tables/model_summary.csv')
    if t08_full_model_summary_expected_sha256 is None or t08_full_model_summary_actual_sha256 != t08_full_model_summary_expected_sha256:
        raise RuntimeError(
            f'T08 FULL tables/model_summary.csv hash does not match its own artifact manifest '
            f'(expected {t08_full_model_summary_expected_sha256}, actual {t08_full_model_summary_actual_sha256}). '
            f'Refusing to reuse unverified evidence.'
        )
    t08_reference_summary = pd.read_csv(t08_full_model_summary_path_ref)
    # Same audit-only label erratum applied in memory (T08's derived table copied the 'random' row
    # verbatim from T07); the stored file on disk is never touched.
    t08_reference_summary = t08_reference_summary.copy()
    t08_reference_summary.loc[t08_reference_summary['ranking_method'] == 'random', 'ranking_method'] = 'random_seed_42'
    tlearner_reference_row = t08_reference_summary.loc[t08_reference_summary['ranking_method'] == 'tlearner'].iloc[0]
    print('Using hash-verified frozen development results (T08 FULL).')
    print(f'  Reproducibility: T08 FULL run_id={T08_FULL_RUN_ID_ACCEPTED}')
else:
    print('RUN_T08_FULL_STAGE = True: T08 FULL will be recomputed from scratch below.')


Using hash-verified frozen development results (T08 FULL).
  Reproducibility: T08 FULL run_id=t08_full_20260818T160823Z_081282


#### Reproducibility note (technical): T08 FULL execution

SMOKE passed every correctness/artifact-mechanism check. Under D30, T08's
approved path is `SMOKE -> FULL` with `resource_gates = 0`, so FULL is the
only remaining stage. FULL fits `mu1` on the complete treated train partition
and `mu0` on the complete control train partition, early-stops each against
its own arm's complete validation subset, and both score the complete common
validation cohort. Held-out remains completely sealed.


*(This section -- T08 FULL -- is already accepted. It is skipped by default on Run All; see `RUN_T08_FULL_STAGE` immediately above. Set it to `True` only to deliberately reproduce T08 FULL from scratch.)*

In [51]:
if RUN_T08_FULL_STAGE:
    from src.tlearner import (
        TLearnerContractError, assert_aligned_predictions, compute_tau,
        partition_by_arm, reconcile_reloaded_tau,
    )
    import src.tlearner as tlearner_module
    import threading
    import time as _time

    t08_config = json.loads((REPO_ROOT / 'configs' / 't08_tlearner.json').read_text(encoding='utf-8'))
    RESOURCE_GATES_T08 = t08_config['scale_gating']['resource_gates']


    class _PeakRSSSampler:
        # Continuous stage-scoped sampler: polls process.memory_info().rss at a
        # fixed interval and tracks the running maximum observed strictly within
        # this sampler's own start()/stop() window -- a genuinely valid mechanism
        # under ADR-experiment-artifacts' resource-measurement rule (distinct from
        # a same-process before/after peak_wset read, which is not).

        def __init__(self, proc, interval_seconds=1.0):
            self._proc = proc
            self._interval = interval_seconds
            self._max_rss_bytes = proc.memory_info().rss
            self._stop_event = threading.Event()
            self._thread = None

        def _run(self):
            while not self._stop_event.is_set():
                rss = self._proc.memory_info().rss
                if rss > self._max_rss_bytes:
                    self._max_rss_bytes = rss
                self._stop_event.wait(self._interval)

        def start(self):
            self._thread = threading.Thread(target=self._run, daemon=True)
            self._thread.start()
            return self

        def stop(self):
            self._stop_event.set()
            if self._thread is not None:
                self._thread.join(timeout=5)
            return self._max_rss_bytes


    t08_full_started = datetime.now(timezone.utc)
    t08_full_wall_start = __import__('time').perf_counter()
    t08_full_baseline_rss_bytes = process.memory_info().rss
    t08_full_rss_sampler = _PeakRSSSampler(process, interval_seconds=1.0).start()

    RUN_ID_T08_FULL = t08_full_started.strftime('t08_full_%Y%m%dT%H%M%SZ_%f')
    RUN_ROOT_T08_FULL = REPO_ROOT / 'outputs' / 'runs' / RUN_ID_T08_FULL
    RUN_ROOT_T08_FULL.mkdir(parents=True, exist_ok=False)

    try:
        t08_full_git_head = subprocess.run(['git', 'rev-parse', 'HEAD'], cwd=REPO_ROOT, check=True, capture_output=True, text=True).stdout.strip()
        t08_full_git_dirty = bool(subprocess.run(['git', 'status', '--porcelain'], cwd=REPO_ROOT, check=True, capture_output=True, text=True).stdout.strip())
    except (OSError, subprocess.CalledProcessError):
        t08_full_git_head, t08_full_git_dirty = None, None

    print(f'RUN_ID_T08_FULL = {RUN_ID_T08_FULL}')


In [52]:
if RUN_T08_FULL_STAGE:
    full_train_frame_t08 = full_frame.loc[full_frame[SOURCE_ROW_ID].isin(train_ids_full)].sort_values(SOURCE_ROW_ID).reset_index(drop=True)
    full_validation_frame_t08 = full_frame.loc[full_frame[SOURCE_ROW_ID].isin(validation_ids_full)].sort_values(SOURCE_ROW_ID).reset_index(drop=True)
    assert len(full_train_frame_t08) == len(train_ids_full)
    assert len(full_validation_frame_t08) == len(validation_ids_full)

    transform_t08_full = IdentityFeatureTransform()
    X_full_train_t08 = transform_t08_full.fit_transform(full_train_frame_t08)
    X_full_validation_t08 = transform_t08_full.transform(full_validation_frame_t08)
    assert_model_feature_contract(X_full_train_t08.columns)
    assert_model_feature_contract(X_full_validation_t08.columns)

    y_full_train_t08 = full_train_frame_t08[PRIMARY_OUTCOME].astype('float64')
    y_full_validation_t08 = full_validation_frame_t08[PRIMARY_OUTCOME].astype('float64')
    t_full_train_t08 = full_train_frame_t08[TREATMENT_COLUMN].astype('float64').to_numpy()
    t_full_validation_t08 = full_validation_frame_t08[TREATMENT_COLUMN].astype('float64').to_numpy()
    y_full_validation_arr_t08 = y_full_validation_t08.to_numpy()
    source_row_id_full_validation_t08 = full_validation_frame_t08[SOURCE_ROW_ID].to_numpy()

    print(f'X_full_train_t08: {X_full_train_t08.shape}, X_full_validation_t08: {X_full_validation_t08.shape}')


### T08.1 Arm partitioning (FULL)

In [53]:
if RUN_T08_FULL_STAGE:
    (X_train_treated_full_t08,), (X_train_control_full_t08,) = partition_by_arm(t_full_train_t08, X_full_train_t08.to_numpy())
    X_train_treated_full_t08 = pd.DataFrame(X_train_treated_full_t08, columns=X_full_train_t08.columns)
    X_train_control_full_t08 = pd.DataFrame(X_train_control_full_t08, columns=X_full_train_t08.columns)
    (y_train_treated_full_t08,), (y_train_control_full_t08,) = partition_by_arm(t_full_train_t08, y_full_train_t08.to_numpy())

    (X_val_treated_full_t08,), (X_val_control_full_t08,) = partition_by_arm(t_full_validation_t08, X_full_validation_t08.to_numpy())
    X_val_treated_full_t08 = pd.DataFrame(X_val_treated_full_t08, columns=X_full_validation_t08.columns)
    X_val_control_full_t08 = pd.DataFrame(X_val_control_full_t08, columns=X_full_validation_t08.columns)
    (y_val_treated_full_t08,), (y_val_control_full_t08,) = partition_by_arm(t_full_validation_t08, y_full_validation_arr_t08)

    print(f'mu1 FULL train rows: {len(X_train_treated_full_t08):,} (early-stop validation: {len(X_val_treated_full_t08):,})')
    print(f'mu0 FULL train rows: {len(X_train_control_full_t08):,} (early-stop validation: {len(X_val_control_full_t08):,})')


### T08.2 Fit mu1 and mu0 (FULL, identical frozen config)

In [54]:
if RUN_T08_FULL_STAGE:
    mu1_model_full = lgb_baseline.fit_binary_classifier(
        X_train_treated_full_t08, y_train_treated_full_t08, X_val_treated_full_t08, y_val_treated_full_t08,
    )
    mu0_model_full = lgb_baseline.fit_binary_classifier(
        X_train_control_full_t08, y_train_control_full_t08, X_val_control_full_t08, y_val_control_full_t08,
    )
    assert mu1_model_full.config_hash == mu0_model_full.config_hash
    print(f'mu1: best_iteration={mu1_model_full.best_iteration}, config_hash={mu1_model_full.config_hash[:16]}...')
    print(f'mu0: best_iteration={mu0_model_full.best_iteration}, config_hash={mu0_model_full.config_hash[:16]}...')


### T08.3 Score the full common validation cohort (FULL)

In [55]:
if RUN_T08_FULL_STAGE:
    mu1_hat_full = lgb_baseline.predict_probabilities(mu1_model_full, X_full_validation_t08)
    mu0_hat_full = lgb_baseline.predict_probabilities(mu0_model_full, X_full_validation_t08)

    assert_aligned_predictions(source_row_id_full_validation_t08, source_row_id_full_validation_t08, source_row_id_full_validation_t08)
    assert len(mu1_hat_full) == len(validation_ids_full) == len(mu0_hat_full)
    assert set(source_row_id_full_validation_t08) == set(validation_ids_full)
    print('Both surfaces scored the full validation cohort:', len(mu1_hat_full), 'rows')


### T08.4 tau_hat = mu1_hat - mu0_hat (FULL)

In [56]:
if RUN_T08_FULL_STAGE:
    tau_hat_full = compute_tau(mu1_hat_full, mu0_hat_full)
    np.testing.assert_array_equal(tau_hat_full, mu1_hat_full - mu0_hat_full)

    positive_count_full = int((tau_hat_full > 0).sum())
    negative_count_full = int((tau_hat_full < 0).sum())
    zero_count_full = int((tau_hat_full == 0).sum())
    assert positive_count_full == int(((mu1_hat_full - mu0_hat_full) > 0).sum())
    print(f'tau_hat (FULL): positive={positive_count_full:,} negative={negative_count_full:,} zero={zero_count_full:,}')


### T08.5 Evaluate via T06 only (FULL)

In [57]:
if RUN_T08_FULL_STAGE:
    tlearner_ranking_full = metrics.evaluate_ranking(
        tau_hat_full, t_full_validation_t08, y_full_validation_arr_t08, source_row_id_full_validation_t08,
    )
    print(f'T-Learner FULL qini_area={tlearner_ranking_full.qini_area:.4f} qini_above_random={tlearner_ranking_full.qini_above_random:.4f}')
    # metrics.random_ranking_reference_distribution() and Response's fit_binary_classifier() are never
    # called here -- T08 FULL does not recompute T07's random reference or Response.


### T08.6 Nuisance diagnostics (each surface, own factual arm only)

In [58]:
if RUN_T08_FULL_STAGE:
    mu1_diag_full = metrics.response_diagnostics(mu1_hat_full[t_full_validation_t08 == 1], y_full_validation_arr_t08[t_full_validation_t08 == 1])
    mu0_diag_full = metrics.response_diagnostics(mu0_hat_full[t_full_validation_t08 == 0], y_full_validation_arr_t08[t_full_validation_t08 == 0])
    print(f'mu1 diagnostics (treated-arm factual): ROC-AUC={mu1_diag_full.roc_auc:.4f} AP={mu1_diag_full.average_precision:.4f} logloss={mu1_diag_full.log_loss:.4f}')
    print(f'mu0 diagnostics (control-arm factual): ROC-AUC={mu0_diag_full.roc_auc:.4f} AP={mu0_diag_full.average_precision:.4f} logloss={mu0_diag_full.log_loss:.4f}')
    # Diagnostic only (D27) -- never a causal ranking claim; distinct from tlearner_ranking_full above.


### T08.7 T07 reuse verification (no recomputation)

In [59]:
if RUN_T08_FULL_STAGE:
    t08_full_t07_reuse_hashes = {}
    for rel_path in ('models/response_model.txt',
                      'predictions/development/response/seed_42/validation_predictions.parquet',
                      'predictions/development/random/seed_42/validation_scores.parquet',
                      'tables/model_summary.csv'):
        actual = hashlib.sha256((t07_full_root / rel_path).read_bytes()).hexdigest()
        expected = t07_artifact_hashes[rel_path]
        t08_full_t07_reuse_hashes[rel_path] = {'expected': expected, 'actual': actual, 'matches': actual == expected}
        assert actual == expected, 'T07 artifact ' + rel_path + ' hash mismatch -- refusing to cite unverified evidence'

    t07_reference_random_row = t07_reference_summary.loc[t07_reference_summary['ranking_method'] == 'random'].iloc[0]
    t07_reference_response_row = t07_reference_summary.loc[t07_reference_summary['ranking_method'] == 'response'].iloc[0]
    t07_model_summary_sha256 = t08_full_t07_reuse_hashes['tables/model_summary.csv']['actual']
    t07_response_predictions_sha256_ref = t08_full_t07_reuse_hashes['predictions/development/response/seed_42/validation_predictions.parquet']['actual']
    t07_random_scores_sha256_ref = t08_full_t07_reuse_hashes['predictions/development/random/seed_42/validation_scores.parquet']['actual']

    print('T07 reuse hash verification (all must match, none recomputed):')
    for path, info in t08_full_t07_reuse_hashes.items():
        print('  ' + path + ': matches=' + str(info['matches']))


### T08.8 Comparative model_summary (random + response + tlearner, explicit lineage)

In [60]:
if RUN_T08_FULL_STAGE:
    def _full_rows_with_run_context_t08(rows, population='full_development_population'):
        for row in rows:
            row = dict(row)
            row.setdefault('run_id', RUN_ID_T08_FULL)
            row.setdefault('stage', 't08_full')
            row.setdefault('population', population)
            yield row


    comparative_model_summary_rows = [
        {
            'ranking_method': 'random',
            'qini_area': float(t07_reference_random_row['qini_area']),
            'qini_above_random': float(t07_reference_random_row['qini_above_random']),
            'theoretical_random_qini_area': float(t07_reference_random_row['theoretical_random_qini_area']),
            'qini_above_random_permutation': float(t07_reference_random_row['qini_above_random_permutation']),
            'source_run_id': T07_FULL_RUN_ID_ACCEPTED,
            'source_prediction_sha256': t07_random_scores_sha256_ref,
            'source_metric_artifact_sha256': t07_model_summary_sha256,
        },
        {
            'ranking_method': 'response',
            'qini_area': float(t07_reference_response_row['qini_area']),
            'qini_above_random': float(t07_reference_response_row['qini_above_random']),
            'theoretical_random_qini_area': float(t07_reference_response_row['theoretical_random_qini_area']),
            'qini_above_random_permutation': float(t07_reference_response_row['qini_above_random_permutation']),
            'source_run_id': T07_FULL_RUN_ID_ACCEPTED,
            'source_prediction_sha256': t07_response_predictions_sha256_ref,
            'source_metric_artifact_sha256': t07_model_summary_sha256,
        },
        {
            'ranking_method': 'tlearner',
            'qini_area': tlearner_ranking_full.qini_area,
            'qini_above_random': tlearner_ranking_full.qini_above_random,
            'theoretical_random_qini_area': tlearner_ranking_full.theoretical_random_qini_area,
            'qini_above_random_permutation': float(t07_reference_random_row['qini_above_random_permutation']),
            'uplift_at_10pct': tlearner_ranking_full.uplift_at_k['10pct'],
            'uplift_at_20pct': tlearner_ranking_full.uplift_at_k['20pct'],
            'uplift_at_30pct': tlearner_ranking_full.uplift_at_k['30pct'],
            'incremental_conversions_at_10pct': tlearner_ranking_full.incremental_conversions_at_k['10pct'],
            'incremental_conversions_at_20pct': tlearner_ranking_full.incremental_conversions_at_k['20pct'],
            'incremental_conversions_at_30pct': tlearner_ranking_full.incremental_conversions_at_k['30pct'],
            'source_run_id': RUN_ID_T08_FULL,
            'source_prediction_sha256': None,  # populated below once the parquet is written and hashed
            'source_metric_artifact_sha256': None,  # self-originated, not reused from elsewhere
        },
    ]
    print('Random/response rows: source_run_id=' + T07_FULL_RUN_ID_ACCEPTED + ' (values inherited unchanged from T07, never recomputed)')
    print('T-Learner row: source_run_id=' + RUN_ID_T08_FULL + ' (genuinely new this run)')


### T08.9 Write FULL artifacts

In [61]:
if RUN_T08_FULL_STAGE:
    tlearner_predictions_frame_full = pd.DataFrame({
        SOURCE_ROW_ID: source_row_id_full_validation_t08,
        'mu1_hat': mu1_hat_full,
        'mu0_hat': mu0_hat_full,
        'tau_hat': tau_hat_full,
    })
    tlearner_predictions_bytes_full = tlearner_predictions_frame_full.to_parquet(index=False)
    write_bytes_new(RUN_ROOT_T08_FULL, 'predictions/development/tlearner/seed_42/validation_predictions.parquet', tlearner_predictions_bytes_full)
    tlearner_predictions_sha256_full = hashlib.sha256(tlearner_predictions_bytes_full).hexdigest()
    comparative_model_summary_rows[2]['source_prediction_sha256'] = tlearner_predictions_sha256_full

    mu1_model_text_full = mu1_model_full.booster.model_to_string()
    mu0_model_text_full = mu0_model_full.booster.model_to_string()
    write_text_new(RUN_ROOT_T08_FULL, 'models/tlearner_mu1.txt', mu1_model_text_full)
    write_text_new(RUN_ROOT_T08_FULL, 'models/tlearner_mu0.txt', mu0_model_text_full)

    uplift_at_k_rows_full_t08 = list(_full_rows_with_run_context_t08([
        {'method': 'tlearner', 'k': label, 'uplift': tlearner_ranking_full.uplift_at_k[label],
         'incremental_conversions': tlearner_ranking_full.incremental_conversions_at_k[label],
         'status': tlearner_ranking_full.top_k_status[label]}
        for label in metrics.RANKING_K_LABELS
    ]))
    tlearner_deciles_rows_full = list(_full_rows_with_run_context_t08(tlearner_ranking_full.decile_table.to_dict('records')))
    response_diagnostics_rows_full_t08 = list(_full_rows_with_run_context_t08([
        {'method': 'tlearner_mu1', **mu1_diag_full.__dict__},
        {'method': 'tlearner_mu0', **mu0_diag_full.__dict__},
    ]))
    model_summary_rows_full_t08 = list(_full_rows_with_run_context_t08(comparative_model_summary_rows))

    for name, rows in (
        ('tables/uplift_at_k.csv', uplift_at_k_rows_full_t08),
        ('tables/tlearner_deciles.csv', tlearner_deciles_rows_full),
        ('tables/response_diagnostics.csv', response_diagnostics_rows_full_t08),
        ('tables/model_summary.csv', model_summary_rows_full_t08),
    ):
        write_text_new(RUN_ROOT_T08_FULL, name, pd.DataFrame(rows).to_csv(index=False, lineterminator='\n'))

    def _flatten_t08_full(value):
        return value if isinstance(value, (str, int, float, bool)) or value is None else json.dumps(value)


    definitions_t08_full = metrics.metric_definitions()
    definitions_sha256_t08_full = hashlib.sha256(json.dumps(definitions_t08_full, sort_keys=True).encode()).hexdigest()
    definitions_rows_t08_full = list(_full_rows_with_run_context_t08(
        [{'field': k, 'value': _flatten_t08_full(v)} for k, v in definitions_t08_full.items()]
        + [{'field': 'definitions_sha256', 'value': definitions_sha256_t08_full}]
    ))
    write_text_new(RUN_ROOT_T08_FULL, 'audit/metric_definitions.csv', pd.DataFrame(definitions_rows_t08_full).to_csv(index=False, lineterminator='\n'))

    print('T08 FULL tables/models/predictions/audit written.')


### T08.10 Figure

In [62]:
if RUN_T08_FULL_STAGE:
    import io as _io_full
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(figsize=(7, 4))
    deciles_full = tlearner_ranking_full.decile_table.sort_values('decile')
    ax.bar(deciles_full['decile'], deciles_full['observed_uplift'])
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_xlabel('Decile (1 = highest tau_hat)')
    ax.set_ylabel('Observed uplift (treated_rate - control_rate)')
    ax.set_title('T-Learner: observed uplift by decile (FULL)')
    fig.tight_layout()
    buffer = _io_full.BytesIO()
    fig.savefig(buffer, format='png', dpi=120, bbox_inches='tight')
    plt.close(fig)
    write_bytes_new(RUN_ROOT_T08_FULL, 'figures/tlearner_uplift_deciles.png', buffer.getvalue())
    print('figures/tlearner_uplift_deciles.png written.')


### T08.11 Reload/reproducibility (FULL, derived tau tolerance)

In [63]:
if RUN_T08_FULL_STAGE:
    reloaded_mu1_booster_full = lgb.Booster(model_str=mu1_model_text_full)
    reloaded_mu0_booster_full = lgb.Booster(model_str=mu0_model_text_full)
    config_hash_matches_full_t08 = (
        lgb_baseline.config_hash() == mu1_model_full.config_hash
        and lgb_baseline.config_hash() == mu0_model_full.config_hash
    )

    X_full_validation_rebuilt_t08 = transform_t08_full.transform(full_validation_frame_t08)
    mu1_reloaded_full = np.asarray(reloaded_mu1_booster_full.predict(X_full_validation_rebuilt_t08, num_iteration=mu1_model_full.best_iteration), dtype=np.float64)
    mu0_reloaded_full = np.asarray(reloaded_mu0_booster_full.predict(X_full_validation_rebuilt_t08, num_iteration=mu0_model_full.best_iteration), dtype=np.float64)

    mu1_reload_matches_full = bool(np.allclose(mu1_reloaded_full, mu1_hat_full, rtol=1e-6, atol=1e-8))
    mu0_reload_matches_full = bool(np.allclose(mu0_reloaded_full, mu0_hat_full, rtol=1e-6, atol=1e-8))
    tau_reload_matches_full, tau_reloaded_full_arr, tau_reload_info_full = reconcile_reloaded_tau(
        mu1_reloaded_full, mu0_reloaded_full, tau_hat_full, mu1_hat_full, mu0_hat_full,
    )
    row_identity_matches_full_t08 = set(source_row_id_full_validation_t08) == set(validation_ids_full)

    reload_verification_t08_full = {
        'config_hash_matches': bool(config_hash_matches_full_t08),
        'mu1_reload_matches_within_tolerance': mu1_reload_matches_full,
        'mu0_reload_matches_within_tolerance': mu0_reload_matches_full,
        'tau_reload_matches_within_derived_tolerance': bool(tau_reload_matches_full),
        'tau_reload_info': tau_reload_info_full,
        'row_identity_matches': bool(row_identity_matches_full_t08),
        'tolerance': {'rtol': 1e-6, 'atol': 1e-8, 'tau_derived_atol': 2e-8},
    }
    print(reload_verification_t08_full)
    assert all([
        reload_verification_t08_full['config_hash_matches'],
        reload_verification_t08_full['mu1_reload_matches_within_tolerance'],
        reload_verification_t08_full['mu0_reload_matches_within_tolerance'],
        reload_verification_t08_full['tau_reload_matches_within_derived_tolerance'],
        reload_verification_t08_full['row_identity_matches'],
    ])


In [64]:
if RUN_T08_FULL_STAGE:
    t08_full_wall_seconds = __import__('time').perf_counter() - t08_full_wall_start
    t08_full_max_rss_bytes_observed = t08_full_rss_sampler.stop()

    t08_full_resource_evidence = {
        'wall_seconds': t08_full_wall_seconds,
        'execution_completed': True,
        'oom_or_termination_observed': False,
        'baseline_rss_bytes': int(t08_full_baseline_rss_bytes),
        'max_rss_bytes_observed': int(t08_full_max_rss_bytes_observed),
        'peak_rss_delta_bytes': int(t08_full_max_rss_bytes_observed) - int(t08_full_baseline_rss_bytes),
        'measurement_mechanism': 'continuous_stage_scoped_sampler_1s_interval',
    }
    print(t08_full_resource_evidence)


In [65]:
if RUN_T08_FULL_STAGE:
    write_json_new(RUN_ROOT_T08_FULL, 'audit/environment.json', {
        'run_id': RUN_ID_T08_FULL,
        'created_at_utc': t08_full_started.isoformat(),
        'git_head': t08_full_git_head,
        'git_dirty': t08_full_git_dirty,
        'python': sys.version,
        'numpy': np.__version__,
        'pandas': pd.__version__,
        'scikit_learn': sklearn.__version__,
        'lightgbm': lgb.__version__,
        'psutil': psutil.__version__,
        'total_ram_bytes': psutil.virtual_memory().total,
    })

    write_json_new(RUN_ROOT_T08_FULL, 'audit/run_config.json', {
        'run_id': RUN_ID_T08_FULL,
        'created_at_utc': t08_full_started.isoformat(),
        'stage': 't08_full',
        'population': 'full_development_population',
        'git_head': t08_full_git_head,
        'git_dirty': t08_full_git_dirty,
        'development_data_accessed': True,
        'held_out_data_accessed': False,
        'notebook_source_sha256': notebook_source_sha256(NOTEBOOK_PATH) if NOTEBOOK_PATH.is_file() else None,
        'src_lightgbm_baseline_sha256': sha256_file(REPO_ROOT / 'src' / 'lightgbm_baseline.py'),
        'src_tlearner_sha256': sha256_file(REPO_ROOT / 'src' / 'tlearner.py'),
        'src_metrics_sha256': sha256_file(REPO_ROOT / 'src' / 'metrics.py'),
        'processed_sha256': processed_sha256,
        'split_membership_sha256': observed_membership_hash,
        't05_run_id': t05_config['lifecycle_state_evidence']['authorizing_run_id'],
        't04_lifecycle_state': t04_config['lifecycle_state'],
        't06_authoritative_run_id': t08_config['input']['metrics_interface']['authoritative_t06_run_id'],
        't06_authoritative_definitions_sha256': t08_config['input']['metrics_interface']['authoritative_t06_definitions_sha256'],
        't07_reuse': {
            'source_run_id': T07_FULL_RUN_ID_ACCEPTED,
            'source_erratum_run_id': T07_ERRATUM_RUN_ID_ACCEPTED,
            'hash_verification': t08_full_t07_reuse_hashes,
            'recomputed': False,
        },
        't08_smoke_reuse': {
            'source_run_id': T08_SMOKE_RUN_ID_ACCEPTED,
            'source_erratum_run_id': T08_SMOKE_ERRATUM_RUN_ID_ACCEPTED,
            'refit': False,
        },
        'train_treated_count': int(len(X_train_treated_full_t08)),
        'train_control_count': int(len(X_train_control_full_t08)),
        'validation_treated_count': int(len(X_val_treated_full_t08)),
        'validation_control_count': int(len(X_val_control_full_t08)),
        'scale_gating': {'policy': 'D30', 'stage': 'FULL', 'resource_gates': RESOURCE_GATES_T08, 'preceding_smoke_run_id': T08_SMOKE_RUN_ID_ACCEPTED},
        'lightgbm_config': lgb_baseline.FROZEN_BINARY_CONFIG,
        'mu1_config_hash': mu1_model_full.config_hash,
        'mu0_config_hash': mu0_model_full.config_hash,
        'mu1_best_iteration': mu1_model_full.best_iteration,
        'mu0_best_iteration': mu0_model_full.best_iteration,
        'lightgbm_num_boost_round_cap': lgb_baseline.NUM_BOOST_ROUND_CAP,
        'lightgbm_early_stopping_rounds': lgb_baseline.EARLY_STOPPING_ROUNDS,
        'tlearner_predictions_sha256': tlearner_predictions_sha256_full,
        'definitions_sha256': definitions_sha256_t08_full,
        'resource_evidence': t08_full_resource_evidence,
        'reload_verification': reload_verification_t08_full,
    })
    print('T08 FULL run_config.json / environment.json written.')


In [66]:
if RUN_T08_FULL_STAGE:
    finalize_artifact_manifest(
        RUN_ROOT_T08_FULL,
        run_id=RUN_ID_T08_FULL,
        final_status='COMPLETED_T08_FULL_VERIFIED',
        created_at_utc=datetime.now(timezone.utc).isoformat(),
        stage='t08_full',
        population='full_development_population',
        external_artifacts=[
            {'path': 'kaggle/02_uplift_modeling.ipynb#sources', 'role': 'human_readable_protocol_source',
             'sha256': notebook_source_sha256(NOTEBOOK_PATH) if NOTEBOOK_PATH.is_file() else None, 'status': 'PASS'},
            {'path': 'src/tlearner.py', 'role': 'reusable_t08_contract', 'sha256': sha256_file(REPO_ROOT / 'src' / 'tlearner.py'), 'status': 'PASS'},
            {'path': 'src/lightgbm_baseline.py', 'role': 'reusable_t07_t08_contract', 'sha256': sha256_file(REPO_ROOT / 'src' / 'lightgbm_baseline.py'), 'status': 'PASS'},
            {'path': 'src/metrics.py', 'role': 'reusable_t06_contract', 'sha256': sha256_file(REPO_ROOT / 'src' / 'metrics.py'), 'status': 'PASS'},
            {'path': f'outputs/runs/{T07_FULL_RUN_ID_ACCEPTED}', 'role': 'reused_t07_evidence', 'sha256': None, 'status': 'PASS'},
            {'path': f'outputs/runs/{T08_SMOKE_RUN_ID_ACCEPTED}', 'role': 'preceding_t08_smoke_run', 'sha256': None, 'status': 'PASS'},
        ],
    )
    print(f'T08 FULL run finalized: {RUN_ID_T08_FULL}')

    immutable_write_refused_t08_full = False
    try:
        write_json_new(RUN_ROOT_T08_FULL, 'audit/should_be_refused.json', {'x': 1})
    except (FileExistsError, DataContractError):
        immutable_write_refused_t08_full = True
    except Exception:
        immutable_write_refused_t08_full = False
    print(f'Immutable-run write refusal verified: {immutable_write_refused_t08_full}')
    assert immutable_write_refused_t08_full


In [67]:
if RUN_T08_FULL_STAGE:
    t08_full_summary = {
        'run_id': RUN_ID_T08_FULL,
        'train_treated_count': int(len(X_train_treated_full_t08)),
        'train_control_count': int(len(X_train_control_full_t08)),
        'validation_treated_count': int(len(X_val_treated_full_t08)),
        'validation_control_count': int(len(X_val_control_full_t08)),
        'mu1_config_hash': mu1_model_full.config_hash,
        'mu0_config_hash': mu0_model_full.config_hash,
        'mu1_best_iteration': mu1_model_full.best_iteration,
        'mu0_best_iteration': mu0_model_full.best_iteration,
        'tau_sign_counts': {'positive': positive_count_full, 'negative': negative_count_full, 'zero': zero_count_full},
        'tlearner_ranking_full': {
            'qini_area': tlearner_ranking_full.qini_area,
            'qini_above_random': tlearner_ranking_full.qini_above_random,
            'theoretical_random_qini_area': tlearner_ranking_full.theoretical_random_qini_area,
            'uplift_at_k': tlearner_ranking_full.uplift_at_k,
            'incremental_conversions_at_k': tlearner_ranking_full.incremental_conversions_at_k,
        },
        'mu1_diagnostics': mu1_diag_full.__dict__,
        'mu0_diagnostics': mu0_diag_full.__dict__,
        'comparison': comparative_model_summary_rows,
        'reload_verification': reload_verification_t08_full,
        't07_reuse_hash_verification': t08_full_t07_reuse_hashes,
        'resource_evidence': t08_full_resource_evidence,
        'immutable_write_refused': immutable_write_refused_t08_full,
    }
    print(json.dumps(t08_full_summary, indent=2, default=str))


## Comparison

Random, Response, and T-Learner are evaluated on the identical frozen
validation cohort, all through the same T06 metric interface -- no method has
its own separate formula. The theoretical random line is the primary no-skill
reference (D11) and has `qini_above_random = 0` by construction, since it is
compared against itself; the seed-42 draw below is one illustrative empirical
realization of a random ranking, not the benchmark itself.


In [68]:
from IPython.display import Markdown, display

_random_row = t07_reference_summary.loc[t07_reference_summary['ranking_method'] == 'random_seed_42'].iloc[0]
_response_row = t07_reference_summary.loc[t07_reference_summary['ranking_method'] == 'response'].iloc[0]

comparison_table = pd.DataFrame([
    {'method': 'theoretical_random (primary reference)', 'qini_area': None, 'qini_above_random': 0.0},
    {'method': 'random_seed_42 (illustrative draw)', 'qini_area': float(_random_row['qini_area']), 'qini_above_random': float(_random_row['qini_above_random'])},
    {'method': 'response', 'qini_area': float(_response_row['qini_area']), 'qini_above_random': float(_response_row['qini_above_random'])},
    {'method': 'tlearner', 'qini_area': float(tlearner_reference_row['qini_area']), 'qini_above_random': float(tlearner_reference_row['qini_above_random'])},
])
print(comparison_table.to_string(index=False))


                                method    qini_area  qini_above_random
theoretical_random (primary reference)          NaN           0.000000
    random_seed_42 (illustrative draw)  1006.179420         -18.489186
                              response -1114.643899       -2139.312505
                              tlearner  -247.411508       -1272.080114


## Interpretation and limitations

In [69]:
display(Markdown(
    f"- **Response** predicts factual conversion well but does not rank uplift well on this cohort: "
    f"`qini_above_random = {float(_response_row['qini_above_random']):.2f}`.\n\n"
    f"- **T-Learner** estimates `tau_hat(x) = mu1_hat(x) - mu0_hat(x)` from two arm-specific factual "
    f"outcome surfaces (`mu1` fit on treated rows only, `mu0` fit on control rows only); treatment is "
    f"never a feature in either surface.\n\n"
    f"- On the frozen validation cohort, T-Learner reaches `qini_above_random = "
    f"{float(tlearner_reference_row['qini_above_random']):.2f}`, against the theoretical random "
    f"reference at exactly 0 by construction.\n\n"
    f"- **This fitted T-Learner did not produce a useful uplift ranking relative to the theoretical "
    f"random reference on this development validation cohort.**\n\n"
    f"- This does not show that treatment heterogeneity is absent -- it is a statement about this "
    f"fitted model's ranking on this cohort, not about the underlying causal structure.\n\n"
    f"- It does not invalidate T-Learner methodology or this implementation.\n\n"
    f"- It does not authorize post-hoc tuning outside the frozen D30/docs-06 protocol -- any future "
    f"model change follows its own CODE PLAN and scale-gating path.\n\n"
    f"- T-Learner's `qini_above_random` is less negative than Response's on this cohort; this is "
    f"reported as a numeric fact only, not a claim that T-Learner is \"better\" in any general sense -- "
    f"both rank below the theoretical no-skill reference here."
))


- **Response** predicts factual conversion well but does not rank uplift well on this cohort: `qini_above_random = -2139.31`.

- **T-Learner** estimates `tau_hat(x) = mu1_hat(x) - mu0_hat(x)` from two arm-specific factual outcome surfaces (`mu1` fit on treated rows only, `mu0` fit on control rows only); treatment is never a feature in either surface.

- On the frozen validation cohort, T-Learner reaches `qini_above_random = -1272.08`, against the theoretical random reference at exactly 0 by construction.

- **This fitted T-Learner did not produce a useful uplift ranking relative to the theoretical random reference on this development validation cohort.**

- This does not show that treatment heterogeneity is absent -- it is a statement about this fitted model's ranking on this cohort, not about the underlying causal structure.

- It does not invalidate T-Learner methodology or this implementation.

- It does not authorize post-hoc tuning outside the frozen D30/docs-06 protocol -- any future model change follows its own CODE PLAN and scale-gating path.

- T-Learner's `qini_above_random` is less negative than Response's on this cohort; this is reported as a numeric fact only, not a claim that T-Learner is "better" in any general sense -- both rank below the theoretical no-skill reference here.